In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
def summarize_gap_resolution(gap_counts: pd.Series, cap_hours: int = 3) -> dict:
    """Given the gap-length -> total-rows Series already produced by
    _missing_gap_length_counts, reports what fraction of DISTINCT gaps
    ('runs') the forward-fill cap actually resolves -- not the fraction
    of missing rows, which is dominated by large structural gaps (e.g.
    the unavoidable t-168 lag startup period) and would understate how
    effective the cap is at fixing genuine short, transient outages."""
    runs_by_length = {length: rows // length for length, rows in gap_counts.items()}
    total_runs = sum(runs_by_length.values())
    resolved_runs = sum(n for length, n in runs_by_length.items() if length <= cap_hours)
    return {
        "total_runs": total_runs,
        "resolved_runs": resolved_runs,
        "pct_runs_resolved": 100 * resolved_runs / total_runs if total_runs else float("nan"),
    }

# Day-Ahead Cross-Border Power Price Spread Forecasting
This python code runs an end-to-end pipeline for forecasting and backtesting day-ahead
electricity price spreads across three Nordic-Continental bidding-zone borders (DK1-DE_LU,
DK1-DK2, DK1-SE3). It pulls hourly price, load, wind/solar, and NTC data from the ENTSO-E
Transparency Platform, engineers day-ahead-causal features, and evaluates six forecasting
models (persistence and seasonal-naive baselines, Elastic Net, LightGBM point/quantile,
and an LSTM) under purged/embargoed walk-forward validation.
Forecasts are then converted into trading signals and backtested under explicit execution
costs, NTC-capacity feasibility limits, BRP imbalance risk, and margin financing, with
dedicated stress-testing and serial-dependence diagnostics on top of the baseline backtest,
and SHAP-based descriptive explainability at the end. Every cell can be run independently
given the outputs of the previous phase, and each phase's assumptions are stated explicitly
in its own docstring rather than left implicit.

SETUP:

Runs on python 3.11

pip install entsoe-py pandas numpy scikit-learn lightgbm torch yfinance matplotlib seaborn holidays shap tqdm

Get your free ENTSO-E API token by registrering at https://transparency.entsoe.eu and then 
emailing transparency@entsoe.eu with "Restful API access" in the subject line, then either:
  - set it as an environment variable: export ENTSOE_API_KEY="your-token"
  - or paste it into API_KEY below

# Data Pipeline and Cleaning

This script pulls hourly data in the time-range of january 1st 2022 to september 30st 2025 from:
  - ENTSO-E Transparency Platform (day-ahead prices, load, wind/solar forecasts, NTC)
  - yfinance (TTF gas futures + EU ETS carbon proxy)

and merges them into a single clean hourly DataFrame, saved to disk,
ready for feature engineering.

In [2]:
import os
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

# -----------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------

API_KEY = os.environ.get("ENTSOE_API_KEY", "PASTE_YOUR_TOKEN_HERE")

ZONES = {
    "DK1": "DK_1",
    "DE_LU": "DE_LU",
    "DK2": "DK_2",
    "SE3": "SE_3",
}


BORDERS = [
    ("DK1", "DE_LU"),
    ("DK1", "DK2"),
    ("DK1", "SE3"),
]


START = pd.Timestamp("2022-01-01", tz="Europe/Copenhagen")
END = pd.Timestamp("2025-09-30", tz="Europe/Copenhagen")

OUTPUT_DIR = Path("./data")
OUTPUT_DIR.mkdir(exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

client = EntsoePandasClient(api_key=API_KEY)


# -----------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------

def _safe_call(fn, *args, label="", retries=3, pause=5, **kwargs):
    """Wrap an entsoe-py call with retries, since the API can be flaky and occasionally rate-limits or times out on large date ranges."""
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except NoMatchingDataError:
            log.warning(f"No data returned for {label} ({args}, {kwargs}) — treating as missing")
            return None
        except Exception as e:
            log.warning(f"[{label}] attempt {attempt}/{retries} failed: {e}")
            if attempt == retries:
                raise
            time.sleep(pause)


def fetch_day_ahead_prices(zone: str) -> pd.Series | None:
    log.info(f"Fetching day-ahead prices for {zone}")
    s = _safe_call(client.query_day_ahead_prices, zone, start=START, end=END, label=f"day_ahead_prices_{zone}")
    if s is None:
        return None
    s.name = f"price_{zone}"
    return s


def fetch_load(zone: str) -> pd.DataFrame | None:
    log.info(f"Fetching load forecast & actual for {zone}")
    df = _safe_call(client.query_load_and_forecast, zone, start=START, end=END, label=f"load_{zone}")
    if df is None:
        return None
    df = df.rename(columns={
        "Forecasted Load": f"load_forecast_{zone}",
        "Actual Load": f"load_actual_{zone}",
    })
    return df


def fetch_wind_solar_forecast(zone: str) -> pd.DataFrame | None:
    log.info(f"Fetching wind & solar forecast for {zone}")
    df = _safe_call(
        client.query_wind_and_solar_forecast, zone, start=START, end=END,
        psr_type=None, label=f"wind_solar_{zone}",
    )
    if df is None:
        return None


    df = df.add_suffix(f"_{zone}")
    return df


def fetch_ntc(zone_from: str, zone_to: str) -> pd.Series | None:
    """Net Transfer Capacity in one direction."""
    log.info(f"Fetching NTC {zone_from} -> {zone_to}")
    s = _safe_call(
        client.query_net_transfer_capacity_dayahead,
        zone_from, zone_to, start=START, end=END,
        label=f"ntc_{zone_from}_{zone_to}",
    )
    if s is None:
        log.warning(
            f"NTC {zone_from}->{zone_to}: no data across the whole date range. "
            "This border's NTC feature will be all-NaN. If unexpected, verify "
            "on the ENTSO-E Transparency Platform whether NTC is published for "
            "this pair, or try query_crossborder_flows as a substitute."
        )
        return None
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    s.name = f"ntc_{zone_from}_to_{zone_to}"
    return s


def fetch_commodity_proxies() -> pd.DataFrame:
    """Daily TTF gas futures + EU ETS carbon proxy via yfinance."""
    import yfinance as yf

    log.info("Fetching commodity proxies (TTF gas, carbon)")
    tickers = {
        "ttf_gas_proxy": "TTF=F",
        "eu_carbon_proxy": "KRBN",
    }

    frames = []
    for name, ticker in tickers.items():
        try:
            hist = yf.download(ticker, start=START.date(), end=END.date(), progress=False)
            if hist.empty:
                log.warning(f"No yfinance data for {ticker} ({name}) — skipping")
                continue


            close = hist["Close"]
            if isinstance(close, pd.DataFrame):
                close = close.squeeze("columns")
            close.name = name
            frames.append(close)
        except Exception as e:
            log.warning(f"Failed to fetch {ticker} ({name}): {e}")
            log.warning("Full traceback:", exc_info=True)

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, axis=1)
    idx = pd.to_datetime(df.index)
    if getattr(idx, "tz", None) is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")
    df.index = idx.tz_convert("Europe/Copenhagen")


    df = df.sort_index().shift(1)
    return df


# -----------------------------------------------------------------------
# MAIN PIPELINE
# -----------------------------------------------------------------------

def build_dataset() -> pd.DataFrame:


    mtu_transition = pd.Timestamp("2025-10-01", tz="Europe/Copenhagen")
    if END >= mtu_transition:
        raise ValueError(
            f"END ({END.date()}) is on or after the SDAC 15-minute MTU transition "
            f"(2025-10-01). Day-ahead data from this date onward is 15-minute "
            "resolution, not hourly, and will not merge cleanly with earlier data. "
            "Either set END before 2025-10-01, or explicitly handle the granularity "
            "change (e.g. resample post-transition data to hourly means) before "
            "removing this check."
        )


    needed_zone_names = sorted({z for pair in BORDERS for z in pair})
    log.info(f"Zones required by BORDERS: {needed_zone_names}")


    price_series = {}
    for zone_name in needed_zone_names:
        code = ZONES[zone_name]
        s = fetch_day_ahead_prices(code)
        if s is None:


            raise RuntimeError(
                f"No day-ahead price data returned for {zone_name} ({code}) across "
                f"{START.date()}..{END.date()}. This zone is required by BORDERS "
                "and the pipeline can't proceed without its price series."
            )
        price_series[zone_name] = s

    prices = pd.concat(price_series.values(), axis=1)


    for zone_a, zone_b in BORDERS:
        code_a, code_b = ZONES[zone_a], ZONES[zone_b]
        prices[f"spread_{zone_a}_{zone_b}"] = prices[f"price_{code_a}"] - prices[f"price_{code_b}"]


    load_frames = [fetch_load(ZONES[z]) for z in needed_zone_names]
    ws_frames = [fetch_wind_solar_forecast(ZONES[z]) for z in needed_zone_names]


    ntc_frames = []
    missing_ntc_cols = []
    for zone_a, zone_b in BORDERS:
        code_a, code_b = ZONES[zone_a], ZONES[zone_b]
        for c_from, c_to in [(code_a, code_b), (code_b, code_a)]:
            s = fetch_ntc(c_from, c_to)
            if s is None:
                missing_ntc_cols.append(f"ntc_{c_from}_to_{c_to}")
            else:
                ntc_frames.append(s)


    commodities = fetch_commodity_proxies()


    frames_to_join = [f for f in (load_frames + ws_frames + ntc_frames) if f is not None]
    n_dropped = len(load_frames) + len(ws_frames) + len(ntc_frames) - len(frames_to_join)
    if n_dropped:
        log.warning(f"{n_dropped} fetch result(s) were empty/missing and excluded from the join "
                    "(see warnings above for which). Their columns will be added back as all-NaN "
                    "after resample so Phase 3 doesn't KeyError on a missing column.")

    log.info("Merging all sources on hourly index")
    df = prices.join(frames_to_join, how="outer")


    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(
            f"Merged index is {type(df.index).__name__}, not DatetimeIndex, after "
            "joining prices with load/wind-solar/NTC frames. This means one of the "
            "join inputs had an incompatible index despite the None-filtering above -- "
            "check for a fetch function returning something other than None on failure."
        )


    df = df.resample("h").mean()


    for col in missing_ntc_cols:
        df[col] = np.nan
        log.warning(f"Added '{col}' as an all-NaN column (no data available for this NTC direction)")


    if not commodities.empty:
        commodities_hourly = commodities.reindex(df.index, method="ffill")
        df = df.join(commodities_hourly)

    return df


def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    log.info("Cleaning dataset")

    before = len(df)

    target_cols = [f"spread_{a}_{b}" for a, b in BORDERS]


    df = df.dropna(subset=target_cols)


    predictor_cols = [c for c in df.columns if c not in target_cols]
    actual_load_cols = [c for c in predictor_cols if c.startswith("load_actual_")]
    if actual_load_cols:
        log.info(
            f"Actual load retained for audit only; excluded from modelling in Phase 3: "
            f"{actual_load_cols}"
        )

    missing_report = df[predictor_cols].isna().mean().sort_values(ascending=False)
    missing_report = missing_report[missing_report > 0]
    if len(missing_report):
        log.info(
            f"Predictor missing-value share passed to Phase 3 (no imputation in Phase 2):\n"
            f"{missing_report}"
        )

    log.info(f"Rows before cleaning: {before}, after: {len(df)}")
    return df


if __name__ == "__main__":
    if API_KEY == "PASTE_YOUR_TOKEN_HERE":
        raise SystemExit(
            "Set your ENTSO-E API key first: export ENTSOE_API_KEY='your-token' "
            "or edit API_KEY at the top of this script."
        )

    raw_df = build_dataset()
    clean_df = clean_dataset(raw_df)

    raw_path = OUTPUT_DIR / "raw_hourly_dataset.parquet"
    clean_path = OUTPUT_DIR / "clean_hourly_dataset.parquet"

    raw_df.to_parquet(raw_path)
    clean_df.to_parquet(clean_path)

    log.info(f"Saved raw dataset -> {raw_path} ({raw_df.shape})")
    log.info(f"Saved clean dataset -> {clean_path} ({clean_df.shape})")
    print(clean_df.tail())


2026-08-20 17:18:04,122 | INFO | Zones required by BORDERS: ['DE_LU', 'DK1', 'DK2', 'SE3']
2026-08-20 17:18:04,122 | INFO | Fetching day-ahead prices for DE_LU
2026-08-20 17:20:12,511 | INFO | Fetching day-ahead prices for DK_1
2026-08-20 17:21:56,099 | INFO | Fetching day-ahead prices for DK_2
2026-08-20 17:23:14,741 | INFO | Fetching day-ahead prices for SE_3
2026-08-20 17:24:35,490 | INFO | Fetching load forecast & actual for DE_LU
2026-08-20 17:26:22,577 | INFO | Fetching load forecast & actual for DK_1
2026-08-20 17:27:33,793 | INFO | Fetching load forecast & actual for DK_2
2026-08-20 17:28:31,407 | INFO | Fetching load forecast & actual for SE_3
2026-08-20 17:29:24,003 | INFO | Fetching wind & solar forecast for DE_LU
2026-08-20 17:31:17,207 | INFO | Fetching wind & solar forecast for DK_1
2026-08-20 17:32:28,026 | INFO | Fetching wind & solar forecast for DK_2
2026-08-20 17:33:20,322 | INFO | Fetching wind & solar forecast for SE_3
2026-08-20 17:33:56,782 | INFO | Fetching NTC 

                           price_DE_LU  price_DK_1  price_DK_2  price_SE_3  \
2025-09-29 18:00:00+00:00       167.60      167.60      167.47      107.41   
2025-09-29 19:00:00+00:00       123.85      123.85      123.79       86.65   
2025-09-29 20:00:00+00:00       108.30      108.30      108.10       44.22   
2025-09-29 21:00:00+00:00        93.68       93.68       93.46       27.00   
2025-09-29 22:00:00+00:00       101.10       97.43      100.85       25.25   

                           spread_DK1_DE_LU  spread_DK1_DK2  spread_DK1_SE3  \
2025-09-29 18:00:00+00:00              0.00            0.13           60.19   
2025-09-29 19:00:00+00:00              0.00            0.06           37.20   
2025-09-29 20:00:00+00:00              0.00            0.20           64.08   
2025-09-29 21:00:00+00:00              0.00            0.22           66.68   
2025-09-29 22:00:00+00:00             -3.67           -3.42           72.18   

                           load_forecast_DE_LU  load_act

# Feature Engineering

Reads the clean hourly dataset produced previously
(./data/clean_hourly_dataset.parquet) and builds the predictive
feature set for EVERY border configured below:

  - Lagged spreads (t-24, t-48, t-168), one set per border
  - Wind forecast differential per border
  - Residual demand per zone (shared across borders that include that zone)
  - Calendar indicators (hour, day of week, month, per-zone holiday dummies)

ZONES / BORDERS must match the previous code block -- duplicated here rather than imported
so each phase script stays independently runnable, same convention as the
original single-border version.

Output: ./data/model_ready_dataset.parquet -- one row per hour, ALL
borders' targets + a shared feature set ready for predictive modeling.

In [3]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

DATA_DIR = Path("./data")
INPUT_PATH = DATA_DIR / "clean_hourly_dataset.parquet"
OUTPUT_PATH = DATA_DIR / "model_ready_dataset.parquet"


ZONES = {
    "DK1": "DK_1",
    "DE_LU": "DE_LU",
    "DK2": "DK_2",
    "SE3": "SE_3",
}
BORDERS = [
    ("DK1", "DE_LU"),
    ("DK1", "DK2"),
    ("DK1", "SE3"),
]


ZONE_COUNTRY = {
    "DK1": "DK",
    "DK2": "DK",
    "DE_LU": "DE",
    "SE3": "SE",
}

TARGET_COLS = [f"spread_{a}_{b}" for a, b in BORDERS]

try:
    import holidays as pyholidays
    HAS_HOLIDAYS_LIB = True
except ImportError:
    HAS_HOLIDAYS_LIB = False
    log.warning(
    )


# -----------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------

def find_col(df: pd.DataFrame, prefix: str, zone_code: str) -> str | None:
    """Column names depend on which series ENTSO-E actually reports for a given zone (see Phase 2 — not every zone reports Wind Offshore separately, for example)."""
    candidates = [c for c in df.columns if c.startswith(prefix) and c.endswith(f"_{zone_code}")]
    if not candidates:
        log.warning(f"No column found for prefix='{prefix}', zone='{zone_code}'")
        return None
    if len(candidates) > 1:
        log.warning(f"Multiple matches for prefix='{prefix}', zone='{zone_code}': {candidates} — using first")
    return candidates[0]


def total_wind(df: pd.DataFrame, zone_code: str) -> pd.Series:
    """Sum onshore + offshore wind if both exist; fall back to whichever is present."""
    onshore_col = find_col(df, "Wind Onshore", zone_code)
    offshore_col = find_col(df, "Wind Offshore", zone_code)

    parts = []
    if onshore_col:
        parts.append(df[onshore_col])
    if offshore_col:
        parts.append(df[offshore_col])

    if not parts:
        log.warning(f"No wind columns found for zone {zone_code} — returning all-NaN series")
        return pd.Series(np.nan, index=df.index)


    total = pd.concat(parts, axis=1).sum(axis=1, min_count=len(parts))
    return total


def solar(df: pd.DataFrame, zone_code: str) -> pd.Series:
    solar_col = find_col(df, "Solar", zone_code)
    if solar_col is None:
        return pd.Series(0.0, index=df.index)
    return df[solar_col]


def load_forecast(df: pd.DataFrame, zone_code: str) -> pd.Series:
    col = f"load_forecast_{zone_code}"
    if col not in df.columns:
        raise KeyError(f"Expected column '{col}' not found — check Phase 2 output")
    return df[col]


# -----------------------------------------------------------------------
# FEATURE BLOCKS
# -----------------------------------------------------------------------

def add_lagged_spreads(df: pd.DataFrame) -> pd.DataFrame:
    log.info("Adding lagged spread features for every border")
    for target in TARGET_COLS:
        df[f"{target}_lag_24"] = df[target].shift(24)
        df[f"{target}_lag_48"] = df[target].shift(48)
        df[f"{target}_lag_168"] = df[target].shift(168)
    return df


def add_wind_forecast_differential(df: pd.DataFrame) -> pd.DataFrame:
    log.info("Adding wind forecast differential per border")


    zone_names = sorted({z for pair in BORDERS for z in pair})
    wind_by_zone = {}
    for zone_name in zone_names:
        code = ZONES[zone_name]
        wind_by_zone[zone_name] = total_wind(df, code)
        df[f"wind_total_{zone_name}"] = wind_by_zone[zone_name]

    for zone_a, zone_b in BORDERS:
        df[f"delta_wind_{zone_a}_{zone_b}"] = wind_by_zone[zone_a] - wind_by_zone[zone_b]
    return df


def add_residual_demand(df: pd.DataFrame) -> pd.DataFrame:
    log.info("Adding residual demand per zone")
    zone_names = sorted({z for pair in BORDERS for z in pair})
    for zone_name in zone_names:
        code = ZONES[zone_name]
        load = load_forecast(df, code)
        wind = total_wind(df, code)
        sol = solar(df, code)
        df[f"residual_demand_{zone_name}"] = load - (wind + sol)
    return df


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    log.info("Adding calendar features")
    idx = df.index

    df["hour"] = idx.hour
    df["day_of_week"] = idx.dayofweek
    df["month"] = idx.month
    df["is_weekend"] = (idx.dayofweek >= 5).astype(int)

    zone_names = sorted({z for pair in BORDERS for z in pair})
    idx_dates = idx.tz_convert("Europe/Copenhagen").date

    if HAS_HOLIDAYS_LIB:
        years = sorted(idx.year.unique().tolist())


        holiday_dummies = {}
        for zone_name in zone_names:
            country = ZONE_COUNTRY[zone_name]
            zone_holidays = pyholidays.country_holidays(country, years=years)
            zone_dates = set(zone_holidays.keys())
            col = f"is_holiday_{zone_name}"
            df[col] = pd.Series(idx_dates, index=idx).isin(zone_dates).astype(int)
            holiday_dummies[zone_name] = df[col]


        for zone_a, zone_b in BORDERS:
            df[f"is_holiday_either_{zone_a}_{zone_b}"] = (
                holiday_dummies[zone_a] | holiday_dummies[zone_b]
            ).astype(int)
    else:
        for zone_name in zone_names:
            df[f"is_holiday_{zone_name}"] = 0
        for zone_a, zone_b in BORDERS:
            df[f"is_holiday_either_{zone_a}_{zone_b}"] = 0


    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    return df


# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------

def build_features() -> pd.DataFrame:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(
            f"{INPUT_PATH} not found — run the Phase 2 pipeline first."
        )

    log.info(f"Loading {INPUT_PATH}")
    df = pd.read_parquet(INPUT_PATH)

    missing_targets = [t for t in TARGET_COLS if t not in df.columns]
    if missing_targets:
        raise KeyError(
            f"Expected target column(s) {missing_targets} not found in {INPUT_PATH}. "
            "Phase 2's BORDERS config must match Phase 3's — check both are in sync."
        )


    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")
    df = df.sort_index()

    full_hourly_index = pd.date_range(
        start=df.index.min().floor("h"),
        end=df.index.max().ceil("h"),
        freq="h",
        tz="UTC",
    )
    rows_before_reindex = len(df)
    df = df.reindex(full_hourly_index)
    inserted = len(df) - rows_before_reindex
    if inserted > 0:
        log.info(
            f"Reindexed to a strict hourly UTC grid: inserted {inserted} missing timestamp row(s) "
            "before lag construction and causal imputation"
        )


    df = add_lagged_spreads(df)
    df = add_wind_forecast_differential(df)
    df = add_residual_demand(df)
    df = add_calendar_features(df)

    return df


def _missing_gap_length_counts(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    """Return counts of missing observations grouped by consecutive-gap length."""
    gap_counts: dict[int, int] = {}

    for col in cols:
        is_missing = df[col].isna().to_numpy()
        if not is_missing.any():
            continue

        starts = np.flatnonzero(is_missing & ~np.r_[False, is_missing[:-1]])
        ends = np.flatnonzero(is_missing & ~np.r_[is_missing[1:], False])
        lengths = ends - starts + 1

        for length in lengths:
            length = int(length)
            gap_counts[length] = gap_counts.get(length, 0) + length

    return pd.Series(gap_counts, dtype="int64").sort_index()


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """Finalize the strictly-hourly model-ready dataset."""
    before = len(df)


    actual_load_cols = [c for c in df.columns if c.startswith("load_actual_")]
    if actual_load_cols:
        log.info(
            f"Dropping realized load columns from model-ready dataset: {actual_load_cols}"
        )
        df = df.drop(columns=actual_load_cols)


    feature_cols_pre = [
        c for c in df.columns
        if c not in TARGET_COLS and not c.startswith("price_")
    ]


    fully_missing = [c for c in feature_cols_pre if df[c].isna().all()]
    if fully_missing:
        log.warning(
            f"Dropping {len(fully_missing)} fully-missing feature column(s): {fully_missing}"
        )
        df = df.drop(columns=fully_missing)
        feature_cols_pre = [c for c in feature_cols_pre if c not in fully_missing]

    remaining_na = df[feature_cols_pre].isna().mean().sort_values(ascending=False)
    remaining_na = remaining_na[remaining_na > 0]

    if len(remaining_na):
        log.info(
            "Applying causal forward-fill to predictor gaps (max 3 consecutive clock hours)"
        )
        log.info(
            f"Predictor columns with missing values before fill:\n{remaining_na}"
        )


        gap_counts = _missing_gap_length_counts(df, feature_cols_pre)
        if len(gap_counts):
            log.info("Missing-gap diagnostics (missing rows by consecutive clock-hour gap length):")
            for gap_len, row_count in gap_counts.items():
                log.info(
                    f"  {gap_len} hour{'s' if gap_len != 1 else ''}: {row_count} rows"
                )
        else:
            log.info("Missing-gap diagnostics: no predictor gaps found")


        df[feature_cols_pre] = df[feature_cols_pre].ffill(limit=3)

        unresolved = df[feature_cols_pre].isna().sum()
        unresolved = unresolved[unresolved > 0]
        if len(unresolved):
            unresolved_rows = df[feature_cols_pre].isna().any(axis=1)
            unresolved_count = int(unresolved_rows.sum())
            log.info(
                f"Dropping rows with unresolved predictor gaps "
                f"(leading gaps or gaps >3 hours): {unresolved.to_dict()}"
            )

            df = df.loc[~unresolved_rows].copy()
            log.info(f"Dropped {unresolved_count} rows with unresolved predictor gaps")
        else:
            log.info("No unresolved predictor gaps remain after causal forward-fill")
    else:
        log.info("No predictor gaps found before causal forward-fill")


    lag_168_cols = [f"{t}_lag_168" for t in TARGET_COLS]
    target_missing_rows = df[TARGET_COLS + lag_168_cols].isna().any(axis=1)
    target_drop_count = int(target_missing_rows.sum())
    df = df.loc[~target_missing_rows].copy()
    log.info(
        f"Dropped {target_drop_count} rows with missing targets or unavailable t-168 lags; "
        f"rows before finalize: {before}, final rows: {len(df)}"
    )


    model_feature_cols = [
        c for c in df.columns
        if c not in TARGET_COLS and not c.startswith("price_")
    ]

    assert df.isna().sum().sum() == 0, (
        "Unexpected NaNs remain after imputation/target filtering — investigate before proceeding"
    )

    log.info(
        f"Final model feature columns ({len(model_feature_cols)}): {model_feature_cols}"
    )
    return df


if __name__ == "__main__":
    feat_df = build_features()
    final_df = finalize(feat_df)

    final_df.to_parquet(OUTPUT_PATH)
    log.info(f"Saved model-ready dataset -> {OUTPUT_PATH} ({final_df.shape})")

    model_feature_cols = [
        c for c in final_df.columns
        if c not in TARGET_COLS and not c.startswith("price_")
    ]
    log.info(f"Feature columns ({len(model_feature_cols)}): {model_feature_cols}")
    print(final_df[TARGET_COLS + model_feature_cols[:8]].tail())


2026-08-20 17:35:53,677 | INFO | Loading data/clean_hourly_dataset.parquet
2026-08-20 17:35:53,776 | INFO | Reindexed to a strict hourly UTC grid: inserted 3 missing timestamp row(s) before lag construction and causal imputation
2026-08-20 17:35:53,776 | INFO | Adding lagged spread features for every border
2026-08-20 17:35:53,781 | INFO | Adding wind forecast differential per border
2026-08-20 17:35:53,796 | WARNING | No column found for prefix='Wind Offshore', zone='SE_3'
2026-08-20 17:35:53,801 | INFO | Adding residual demand per zone
2026-08-20 17:35:53,814 | WARNING | No column found for prefix='Wind Offshore', zone='SE_3'
2026-08-20 17:35:53,817 | INFO | Adding calendar features
2026-08-20 17:35:53,979 | INFO | Dropping realized load columns from model-ready dataset: ['load_actual_DE_LU', 'load_actual_DK_1', 'load_actual_DK_2', 'load_actual_SE_3']
2026-08-20 17:35:53,984 | WARNING | Dropping 4 fully-missing feature column(s): ['ntc_DK_1_to_DK_2', 'ntc_DK_2_to_DK_1', 'ntc_DK_1_to_

                           spread_DK1_DE_LU  spread_DK1_DK2  spread_DK1_SE3  \
2025-09-29 18:00:00+00:00              0.00            0.13           60.19   
2025-09-29 19:00:00+00:00              0.00            0.06           37.20   
2025-09-29 20:00:00+00:00              0.00            0.20           64.08   
2025-09-29 21:00:00+00:00              0.00            0.22           66.68   
2025-09-29 22:00:00+00:00             -3.67           -3.42           72.18   

                           load_forecast_DE_LU  load_forecast_DK_1  \
2025-09-29 18:00:00+00:00           59507.4375              2920.0   
2025-09-29 19:00:00+00:00           56297.0200              2870.0   
2025-09-29 20:00:00+00:00           52638.6125              2863.0   
2025-09-29 21:00:00+00:00           48958.2200              2837.0   
2025-09-29 22:00:00+00:00           48958.2200              2837.0   

                           load_forecast_DK_2  load_forecast_SE_3  \
2025-09-29 18:00:00+00:00          

## Information-set convention

**Forecasting decision:** the target is the day-ahead spread for delivery hour *t*. ENTSO-E forecast products are treated as day-ahead forecasts for the delivery hour and are assumed available before the auction. The dataset does not contain forecast-vintage publication timestamps, so this availability is an explicit assumption/limitation rather than a claim that the raw API response itself proves publication timing. Realized load is excluded from model inputs. Realized spread is used only after the fact for evaluation and, in Phase 5, for risk estimation with a 24-hour lag.

**Commodity timing:** Yahoo Finance observations are conservatively shifted by one observed market session because exact publication timestamps are not available.


**Border selection note:** DK1 is directly interconnected with SE3 via the Konti-Skan HVDC cable, not with SE4 (that link is DK2<->SE4 via the Öresund connector). DK1-SE3 is used as the third border so the comparison stays anchored on a real physical interconnector.

# Predictive Modeling

Predictive Modeling
Cross-Border Power Price Spread Forecasting (Nordic system: DK1 <-> DE-LU, DK1 <-> DK2, DK1 <-> SE3)

Forecast evaluation uses expanding-window walk-forward validation with a 168-hour
outer embargo. Inner model selection uses the same 168-hour gap. Models:
  1. Persistence (t-24)
  2. Seasonal naive (t-168)
  3. Elastic Net + interactions
  4. LightGBM point forecast
  5. LightGBM quantile forecasts (10/50/90)
  6. LSTM (sequence-to-one, 168h lookback)

Outputs per border:
  - phase4_fold_metrics_{zone_a}_{zone_b}.csv
  - phase4_test_predictions_{zone_a}_{zone_b}.csv
  - phase4_dm_tests_{zone_a}_{zone_b}.csv
  - final model files (LightGBM native format; LSTM as a PyTorch state_dict
    + a pickled feature scaler, since inference requires the exact same
    scaling used at training time)

NOTE ON THE LSTM:
This is a tabular walk-forward pipeline, not a sequence pipeline, so the LSTM
needs a sequence-construction layer the other models don't: each prediction
at row p is made from a (LOOKBACK_HOURS, n_features) window of the
LOOKBACK_HOURS rows strictly BEFORE p (df.iloc[p-LOOKBACK_HOURS:p]), never
including row p itself. Two things matter for correctness here and are
handled explicitly below (see make_windowed_arrays and
_embargoed_position_split):
  1. Windows for a fold's TEST predictions are allowed to look back into rows
     that were embargoed OUT of that fold's TRAINING set. This is correct,
     not leakage -- the embargo exists to stop the model being FIT on
     boundary-adjacent rows, not to hide genuinely past data from a
     prediction at inference time. A live deployed model would have exactly
     this data available.
  2. The internal fit/validation split used for early stopping re-applies the
     same EMBARGO_HOURS gap used everywhere else in this file, and windows
     for the fit portion and validation portion are built independently so a
     validation window can never pull feature rows from inside the embargo
     gap (which would reintroduce the leakage the tabular models' internal
     split already avoids).
Because the model-ready dataset is a strict hourly grid (Phase 3), a
LOOKBACK_HOURS-row window is genuinely LOOKBACK_HOURS clock hours, consistent
with how the *_lag_24/48/168 features are constructed elsewhere.


In [4]:
"""Phase 4: Predictive Modeling"""

import logging
import pickle
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

    class tqdm:
        def __init__(self, iterable=None, total=None, desc=None, **kwargs):
            self.iterable = iterable
            self.desc = desc

        def __iter__(self):
            return iter(self.iterable) if self.iterable is not None else iter([])

        def set_postfix(self, *a, **k):
            pass

        def set_description(self, desc):
            self.desc = desc

        def update(self, n=1):
            pass

        def close(self):
            pass

        def __enter__(self):
            return self

        def __exit__(self, *exc):
            self.close()

        @staticmethod
        def write(msg):
            print(msg)

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

try:
    from IPython.display import display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)


@contextmanager
def _timed(label: str):
    """Prints how long a block took, via tqdm.write so it never garbles an active progress bar."""
    t0 = time.time()
    tqdm.write(f"  -> starting {label}...")
    yield
    tqdm.write(f"  -> {label} done in {time.time() - t0:.1f}s")

DATA_DIR = Path("./data")
MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)
INPUT_PATH = DATA_DIR / "model_ready_dataset.parquet"

ZONES = {"DK1": "DK_1", "DE_LU": "DE_LU", "DK2": "DK_2", "SE3": "SE_3"}
BORDERS = [("DK1", "DE_LU"), ("DK1", "DK2"), ("DK1", "SE3")]
TARGET_COLS = [f"spread_{a}_{b}" for a, b in BORDERS]

N_FOLDS = 5
MIN_TRAIN_FRAC = 0.4
EMBARGO_HOURS = 168
EN_CV_SPLITS = 5
QUANTILES = [0.1, 0.5, 0.9]
DIRECTIONAL_EVAL_THRESHOLD = 1.0
SEED = 42

LOOKBACK_HOURS = 168

SEQ_HIDDEN_SIZE = 32
SEQ_NUM_LAYERS = 2
SEQ_DROPOUT = 0.1
SEQ_BATCH_SIZE = 256
SEQ_MAX_EPOCHS = 25
SEQ_PATIENCE = 6

SEQ_LR = 1e-3
SEQ_VAL_FRAC = 0.15
FINAL_SEQ_MAX_EPOCHS = 25

if HAS_TORCH:
    torch.manual_seed(SEED)


# -----------------------------------------------------------------------
# STYLED TABLE DISPLAY
# -----------------------------------------------------------------------

def style_table(df: pd.DataFrame, caption: str = None, money_cols: list[str] = None,
                pct_cols: list[str] = None, highlight_max_cols: list[str] = None,
                highlight_min_cols: list[str] = None, decimals: int = 3):
    """Formats a DataFrame for readable notebook display: currency/percentage
    formatting, a caption, and green/red highlighting of the best/worst value
    per column. Falls back to a plain formatted print if IPython isn't available
    (e.g. running as a plain script rather than in a notebook)."""
    money_cols = money_cols or []
    pct_cols = pct_cols or []
    highlight_max_cols = highlight_max_cols or []
    highlight_min_cols = highlight_min_cols or []

    fmt = {}
    for c in df.columns:
        if c in money_cols:
            fmt[c] = "€{:,.0f}"
        elif c in pct_cols:
            fmt[c] = "{:.1%}"
        elif pd.api.types.is_float_dtype(df[c]):
            fmt[c] = f"{{:.{decimals}f}}"

    styler = df.style.format(fmt, na_rep="-").hide(axis="index")
    if caption:
        styler = styler.set_caption(caption)
    for c in highlight_max_cols:
        if c in df.columns:
            styler = styler.highlight_max(subset=[c], color="#c6efce")
    for c in highlight_min_cols:
        if c in df.columns:
            styler = styler.highlight_min(subset=[c], color="#ffc7ce")

    if HAS_IPYTHON:
        display(styler)
    else:
        print(f"\n{caption or ''}")
        print(df.to_string(index=False))
    return styler

# -----------------------------------------------------------------------
# FEATURES / INTERACTIONS
# -----------------------------------------------------------------------

def get_feature_columns(df: pd.DataFrame) -> list[str]:
    exclude = set(TARGET_COLS) | {f"price_{code}" for code in ZONES.values()}
    return [c for c in df.columns if c not in exclude]


def add_interaction_terms(df: pd.DataFrame, base_features: list[str],
                          zone_a: str, zone_b: str) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()
    new_cols = []
    code_a, code_b = ZONES[zone_a], ZONES[zone_b]
    wind_diff_col = f"delta_wind_{zone_a}_{zone_b}"
    ntc_col = f"ntc_{code_a}_to_{code_b}"
    resdem_a_col = f"residual_demand_{zone_a}"
    resdem_b_col = f"residual_demand_{zone_b}"

    if ntc_col not in df.columns:
        log.warning(f"'{ntc_col}' not in dataset — skipping NTC-based interactions for {zone_a}-{zone_b}")
    else:
        if wind_diff_col in df.columns:
            df["interact_wind_ntc"] = df[wind_diff_col] * df[ntc_col]
            new_cols.append("interact_wind_ntc")
        if resdem_a_col in df.columns and resdem_b_col in df.columns:
            df["residual_demand_diff"] = df[resdem_a_col] - df[resdem_b_col]
            df["interact_resdemand_ntc"] = df["residual_demand_diff"] * df[ntc_col]
            new_cols.extend(["residual_demand_diff", "interact_resdemand_ntc"])
    return df, base_features + new_cols

# -----------------------------------------------------------------------
# WALK-FORWARD SPLITS
# -----------------------------------------------------------------------

def make_walk_forward_folds(n_rows: int, n_folds: int = N_FOLDS,
                            min_train_frac: float = MIN_TRAIN_FRAC,
                            embargo_hours: int = EMBARGO_HOURS):
    min_train_size = int(n_rows * min_train_frac)
    remaining = n_rows - min_train_size
    test_size = remaining // n_folds
    folds = []
    train_end = min_train_size
    for _ in range(n_folds):
        test_start = train_end
        test_end = min(test_start + test_size, n_rows)
        if test_start >= n_rows:
            break
        embargoed_train_end = max(0, train_end - embargo_hours)
        folds.append((slice(0, embargoed_train_end), slice(test_start, test_end)))
        train_end = test_end
    return folds

# -----------------------------------------------------------------------
# METRICS
# -----------------------------------------------------------------------

def directional_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.sign(y_true) == np.sign(y_pred)))


def directional_accuracy_threshold(y_true: np.ndarray, y_pred: np.ndarray,
                                   threshold: float = DIRECTIONAL_EVAL_THRESHOLD) -> float:
    mask = np.abs(y_true) > threshold
    if not mask.any():
        return float("nan")
    return float(np.mean(np.sign(y_true[mask]) == np.sign(y_pred[mask])))


def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if not mask.all():
        y_true, y_pred = y_true[mask], y_pred[mask]
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "directional_accuracy": directional_accuracy(y_true, y_pred),
        "directional_accuracy_gt_1": directional_accuracy_threshold(y_true, y_pred),
    }


def pinball_loss(y_true: np.ndarray, y_pred: np.ndarray, quantile: float) -> float:
    """The loss LightGBM's quantile objective actually optimizes. Asymmetric:
    for a low quantile (e.g. q10), predicting too HIGH is penalized more than
    predicting too low, and vice versa for a high quantile -- unlike RMSE,
    which treats over- and under-prediction identically regardless of which
    quantile is being estimated."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    e = y_true - y_pred
    return float(np.mean(np.maximum(quantile * e, (quantile - 1) * e)))


def evaluate_quantile(y_true: np.ndarray, y_pred: np.ndarray, quantile: float) -> dict:
    """Metrics appropriate for a quantile forecast: pinball loss (the
    quantile-appropriate score) plus RMSE/MAE for reference/comparability
    with the point-forecast models, and directional accuracy since even a
    quantile forecast's sign is used downstream (confidence-weighted sizing
    in Phase 5)."""
    base = evaluate(y_true, y_pred)
    base["pinball_loss"] = pinball_loss(y_true, y_pred, quantile)
    return base


def compute_quantile_calibration(predictions_df: pd.DataFrame) -> dict:
    """Empirical coverage check for the [q10, q90] interval: what fraction of
    realized spreads actually fell below q10, above q90, and inside the
    interval, across the full walk-forward test period (all folds
    concatenated). A well-calibrated model should show close to 10% / 10% /
    80% respectively. This has been an outstanding item on this project's
    limitations list -- the confidence-weighted trading strategy in Phase 5
    is only as trustworthy as this calibration check says it is."""
    if not {"lightgbm_q10", "lightgbm_q90", "y_true"}.issubset(predictions_df.columns):
        return {}
    y = predictions_df["y_true"].to_numpy(dtype=float)
    q10 = predictions_df["lightgbm_q10"].to_numpy(dtype=float)
    q90 = predictions_df["lightgbm_q90"].to_numpy(dtype=float)
    mask = np.isfinite(y) & np.isfinite(q10) & np.isfinite(q90)
    y, q10, q90 = y[mask], q10[mask], q90[mask]
    return {
        "n_observations": int(mask.sum()),
        "empirical_coverage_below_q10": float(np.mean(y < q10)),
        "target_coverage_below_q10": 0.10,
        "empirical_coverage_above_q90": float(np.mean(y > q90)),
        "target_coverage_above_q90": 0.10,
        "empirical_coverage_within_interval": float(np.mean((y >= q10) & (y <= q90))),
        "target_coverage_within_interval": 0.80,
    }


def dm_test(y_true: np.ndarray, pred_1: np.ndarray, pred_2: np.ndarray,
            max_lag: int = 24) -> dict:
    """Newey-West / HAC Diebold-Mariano test for equal predictive accuracy."""
    y = np.asarray(y_true, dtype=float)
    e1 = np.asarray(pred_1, dtype=float)
    e2 = np.asarray(pred_2, dtype=float)
    d = (y - e1) ** 2 - (y - e2) ** 2
    d = d[np.isfinite(d)]
    n = len(d)
    if n < max(20, max_lag + 5):
        return {"dm_stat": np.nan, "p_value": np.nan, "n": n, "horizon_lag": max_lag}
    centered = d - d.mean()
    gamma0 = np.mean(centered ** 2)
    q = min(max_lag, n - 1)
    long_run = gamma0
    for k in range(1, q + 1):
        weight = 1.0 - k / (q + 1.0)
        cov = np.mean(centered[k:] * centered[:-k])
        long_run += 2.0 * weight * cov
    if long_run <= 0:
        return {"dm_stat": np.nan, "p_value": np.nan, "n": n, "horizon_lag": q}
    dm_stat = d.mean() / np.sqrt(long_run / n)
    from math import erf, sqrt
    p_value = 2.0 * (1.0 - 0.5 * (1.0 + erf(abs(dm_stat) / sqrt(2.0))))
    return {"dm_stat": float(dm_stat), "p_value": float(p_value), "n": n, "horizon_lag": q}

# -----------------------------------------------------------------------
# TABULAR MODELS
# -----------------------------------------------------------------------

def fit_persistence(df: pd.DataFrame, test_idx: slice, target_col: str) -> np.ndarray:
    return df[f"{target_col}_lag_24"].iloc[test_idx].values


def fit_seasonal_naive(df: pd.DataFrame, test_idx: slice, target_col: str) -> np.ndarray:
    return df[f"{target_col}_lag_168"].iloc[test_idx].values


def fit_elastic_net(X_train, y_train, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    model = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        alphas=100,
        cv=TimeSeriesSplit(n_splits=EN_CV_SPLITS, gap=EMBARGO_HOURS),
        max_iter=20000,
        tol=1e-5,
        n_jobs=-1,
    )
    model.fit(X_train_s, y_train)
    return model.predict(X_test_s), model


def _lgb_params(objective, metric):
    return {
        "objective": objective,
        "metric": metric,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_data_in_leaf": 20,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "seed": SEED,
        "feature_fraction_seed": SEED,
        "bagging_seed": SEED,
        "data_random_seed": SEED,
        "deterministic": True,
        "force_col_wise": True,
        "verbose": -1,
    }


def _split_for_validation(X_train, y_train, y_val_split=0.15):
    n_val = max(int(len(X_train) * y_val_split), EMBARGO_HOURS + 1)
    if len(X_train) <= n_val + EMBARGO_HOURS:
        raise ValueError("Training window too short for validation plus embargo.")
    val_start = len(X_train) - n_val
    fit_end = val_start - EMBARGO_HOURS
    return X_train.iloc[:fit_end], X_train.iloc[val_start:], y_train.iloc[:fit_end], y_train.iloc[val_start:]


def fit_lightgbm_point(X_train, y_train, X_test, y_val_split=0.15):
    X_fit, X_val, y_fit, y_val = _split_for_validation(X_train, y_train, y_val_split)
    train_set = lgb.Dataset(X_fit, label=y_fit)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)
    params = _lgb_params("regression", "rmse")
    model = lgb.train(
        params, train_set, valid_sets=[val_set], num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    return model.predict(X_test, num_iteration=model.best_iteration), model


def fit_lightgbm_quantile(X_train, y_train, X_test, quantile: float, y_val_split=0.15):
    X_fit, X_val, y_fit, y_val = _split_for_validation(X_train, y_train, y_val_split)
    train_set = lgb.Dataset(X_fit, label=y_fit)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)
    params = _lgb_params("quantile", "quantile")
    params["alpha"] = quantile
    model = lgb.train(
        params, train_set, valid_sets=[val_set], num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    return model.predict(X_test, num_iteration=model.best_iteration), model


# -----------------------------------------------------------------------
# SEQUENCE MODEL (LSTM)
# -----------------------------------------------------------------------

class LSTMRegressor(nn.Module):
    """Sequence-to-one LSTM: consumes a (batch, lookback, n_features) window and
    predicts a single scalar spread value for the hour immediately following it."""

    def __init__(self, n_features: int, hidden_size: int = SEQ_HIDDEN_SIZE,
                 num_layers: int = SEQ_NUM_LAYERS, dropout: float = SEQ_DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]
        return self.head(last_hidden).squeeze(-1)


def make_windowed_arrays(df: pd.DataFrame, feature_cols: list[str], target_col: str,
                          positions: np.ndarray, lookback: int = LOOKBACK_HOURS):
    """Build (lookback, n_features) windows for every position in `positions`."""
    feat_values = df[feature_cols].to_numpy(dtype=np.float32)
    target_values = df[target_col].to_numpy(dtype=np.float32)

    positions = np.asarray(positions)
    valid_mask = positions >= lookback
    n_dropped = int((~valid_mask).sum())
    if n_dropped:
        log.warning(f"Dropping {n_dropped} position(s) with insufficient lookback history "
                    f"(need {lookback} prior rows) -- expected only at the very start of the dataset")
    kept = positions[valid_mask]

    if len(kept) == 0:
        return np.empty((0, lookback, len(feature_cols)), dtype=np.float32), \
               np.empty((0,), dtype=np.float32), kept

    X = np.stack([feat_values[p - lookback:p] for p in kept])
    y = target_values[kept]
    return X, y, kept


def _embargoed_position_split(train_positions: np.ndarray, y_val_split: float = SEQ_VAL_FRAC,
                               embargo_hours: int = EMBARGO_HOURS):
    """Position-array equivalent of _split_for_validation, for the sequence models."""
    n = len(train_positions)
    n_val = max(int(n * y_val_split), embargo_hours + 1)
    if n <= n_val + embargo_hours:
        raise ValueError("Training window too short for validation plus embargo.")
    val_start = n - n_val
    fit_end = val_start - embargo_hours
    return train_positions[:fit_end], train_positions[val_start:]


def _train_sequence_model(model: "nn.Module", X_fit, y_fit, X_val, y_val,
                          max_epochs: int = SEQ_MAX_EPOCHS, patience: int = SEQ_PATIENCE,
                          lr: float = SEQ_LR, batch_size: int = SEQ_BATCH_SIZE,
                          desc: str = "training") -> "nn.Module":
    """Generic training loop with early stopping on validation MSE."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    fit_ds = TensorDataset(torch.from_numpy(X_fit), torch.from_numpy(y_fit))
    fit_loader = DataLoader(fit_ds, batch_size=batch_size, shuffle=True)
    X_val_t = torch.from_numpy(X_val).to(device)
    y_val_t = torch.from_numpy(y_val).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    epoch_bar = tqdm(range(max_epochs), desc=desc, unit="epoch", leave=False)
    for epoch in epoch_bar:
        model.train()
        for xb, yb in fit_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t)
            val_loss = loss_fn(val_preds, y_val_t).item()

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        epoch_bar.set_postfix(val_loss=f"{val_loss:.3f}", best=f"{best_val_loss:.3f}",
                              patience=f"{epochs_without_improvement}/{patience}")

        if epochs_without_improvement >= patience:
            epoch_bar.set_description(f"{desc} (early stopped @ epoch {epoch + 1})")
            break
    epoch_bar.close()

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to("cpu")


def _fit_sequence_model_generic(architecture: str, df: pd.DataFrame, feature_cols: list[str],
                                 target_col: str, train_idx: slice, test_idx: slice):
    """Windowing/scaling/training/prediction logic for the LSTM sequence model."""
    n_rows = len(df)
    train_positions = np.arange(*train_idx.indices(n_rows))
    test_positions = np.arange(*test_idx.indices(n_rows))

    fit_positions, val_positions = _embargoed_position_split(train_positions)

    scaler = StandardScaler()
    scaler.fit(df[feature_cols].iloc[fit_positions[0]:fit_positions[-1] + 1].to_numpy())

    def scaled_windows(positions):
        X, y, kept = make_windowed_arrays(df, feature_cols, target_col, positions)
        if len(kept) == 0:
            return X, y, kept
        n, lb, nf = X.shape
        X_scaled = scaler.transform(X.reshape(-1, nf)).reshape(n, lb, nf).astype(np.float32)
        return X_scaled, y, kept

    X_fit, y_fit, _ = scaled_windows(fit_positions)
    X_val, y_val, _ = scaled_windows(val_positions)
    X_test, y_test_unused, kept_test_positions = scaled_windows(test_positions)

    n_features = len(feature_cols)
    if architecture == "lstm":
        model = LSTMRegressor(n_features=n_features)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    fold_label = f"{architecture} [{target_col}] test@row{test_positions[0]}"
    model = _train_sequence_model(model, X_fit, y_fit, X_val, y_val, desc=fold_label)

    model.eval()
    with torch.no_grad():
        preds_kept = model(torch.from_numpy(X_test)).numpy() if len(kept_test_positions) else np.empty(0)

    full_preds = np.full(len(test_positions), np.nan, dtype=np.float32)
    kept_offsets = np.searchsorted(test_positions, kept_test_positions)
    full_preds[kept_offsets] = preds_kept

    return full_preds, model, scaler


def fit_lstm(df, feature_cols, target_col, train_idx, test_idx):
    return _fit_sequence_model_generic("lstm", df, feature_cols, target_col, train_idx, test_idx)

# -----------------------------------------------------------------------
# WALK-FORWARD LOOP
# -----------------------------------------------------------------------

def run_walk_forward(df: pd.DataFrame, feature_cols: list[str], target_col: str):
    folds = make_walk_forward_folds(len(df))
    log.info(f"[{target_col}] Running {len(folds)} embargoed walk-forward folds "
             f"(min train size {int(len(df) * MIN_TRAIN_FRAC)} rows, embargo {EMBARGO_HOURS}h)")
    all_metrics, all_fold_predictions = [], []

    fold_bar = tqdm(list(enumerate(folds)), desc=f"[{target_col}] folds", unit="fold")
    for fold_i, (train_idx, test_idx) in fold_bar:
        fold_bar.set_postfix(fold=f"{fold_i + 1}/{len(folds)}")
        train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
        X_train, y_train = train_df[feature_cols], train_df[target_col]
        X_test, y_test = test_df[feature_cols], test_df[target_col]
        tqdm.write(f"[{target_col}] Fold {fold_i}: train rows={len(X_train)}, test rows={len(X_test)}")
        fold_preds = {"y_true": y_test.values}

        for name, pred in [
            ("persistence", fit_persistence(df, test_idx, target_col)),
            ("seasonal_naive", fit_seasonal_naive(df, test_idx, target_col)),
        ]:
            fold_preds[name] = pred
            all_metrics.append({"fold": fold_i, "model": name, **evaluate(y_test.values, pred)})

        with _timed(f"fold {fold_i} elastic_net"):
            en_preds, _ = fit_elastic_net(X_train, y_train, X_test)
        fold_preds["elastic_net"] = en_preds
        all_metrics.append({"fold": fold_i, "model": "elastic_net", **evaluate(y_test.values, en_preds)})

        if HAS_LIGHTGBM:
            with _timed(f"fold {fold_i} lightgbm_point"):
                lgb_preds, _ = fit_lightgbm_point(X_train, y_train, X_test)
            fold_preds["lightgbm_point"] = lgb_preds
            all_metrics.append({"fold": fold_i, "model": "lightgbm_point", **evaluate(y_test.values, lgb_preds)})

            with _timed(f"fold {fold_i} lightgbm_quantiles"):
                for q in QUANTILES:
                    q_preds, _ = fit_lightgbm_quantile(X_train, y_train, X_test, q)
                    fold_preds[f"lightgbm_q{int(q*100)}"] = q_preds
                    # All three quantiles are now evaluated (previously only q50 was).
                    all_metrics.append({"fold": fold_i, "model": f"lightgbm_q{int(q*100)}",
                                        **evaluate_quantile(y_test.values, q_preds, q)})
        else:
            log.warning("lightgbm not installed — skipping LightGBM execution.")

        if HAS_TORCH:
            with _timed(f"fold {fold_i} lstm (see epoch progress bar)"):
                lstm_preds, _, _ = fit_lstm(df, feature_cols, target_col, train_idx, test_idx)
            fold_preds["lstm"] = lstm_preds
            all_metrics.append({"fold": fold_i, "model": "lstm", **evaluate(y_test.values, lstm_preds)})
        else:
            log.warning("torch not installed — skipping LSTM execution.")

        fold_df = pd.DataFrame(fold_preds, index=test_df.index)
        fold_df["fold"] = fold_i
        all_fold_predictions.append(fold_df)
    fold_bar.close()

    metrics_df = pd.DataFrame(all_metrics)
    predictions_df = pd.concat(all_fold_predictions, axis=0) if all_fold_predictions else pd.DataFrame()

    dm_rows = []
    if len(predictions_df):
        baseline = predictions_df["persistence"].to_numpy()
        non_model_cols = {"y_true", "fold", "persistence", "seasonal_naive",
                           "lightgbm_q10", "lightgbm_q50", "lightgbm_q90"}
        for model in [c for c in predictions_df.columns if c not in non_model_cols]:
            res = dm_test(predictions_df["y_true"].to_numpy(), predictions_df[model].to_numpy(), baseline, max_lag=24)
            dm_rows.append({"model_1": model, "model_2": "persistence", **res})
        if "seasonal_naive" in predictions_df.columns:
            res = dm_test(predictions_df["y_true"].to_numpy(), predictions_df["seasonal_naive"].to_numpy(), baseline, max_lag=24)
            dm_rows.append({"model_1": "seasonal_naive", "model_2": "persistence", **res})
    dm_df = pd.DataFrame(dm_rows)

    calibration = compute_quantile_calibration(predictions_df)

    return metrics_df, predictions_df, dm_df, calibration


FINAL_NUM_BOOST_ROUNDS = 500


def fit_final_models(df: pd.DataFrame, feature_cols: list[str], target_col: str,
                     zone_a: str, zone_b: str):
    """Refit final models on the full available sample."""
    suffix = f"{zone_a}_{zone_b}"

    if HAS_LIGHTGBM:
        X, y = df[feature_cols], df[target_col]
        train_set = lgb.Dataset(X, label=y)

        point_model = lgb.train(_lgb_params("regression", "rmse"), train_set, num_boost_round=FINAL_NUM_BOOST_ROUNDS)
        point_model.save_model(str(MODEL_DIR / f"lightgbm_point_final_{suffix}.txt"))

        for q in QUANTILES:
            params = _lgb_params("quantile", "quantile")
            params["alpha"] = q
            q_model = lgb.train(params, train_set, num_boost_round=FINAL_NUM_BOOST_ROUNDS)
            q_model.save_model(str(MODEL_DIR / f"lightgbm_quantile_{int(q*100)}_final_{suffix}.txt"))

    if HAS_TORCH:
        n_rows = len(df)
        all_positions = np.arange(n_rows)
        fit_positions, val_positions = _embargoed_position_split(all_positions, y_val_split=SEQ_VAL_FRAC)

        scaler = StandardScaler()
        scaler.fit(df[feature_cols].iloc[fit_positions[0]:fit_positions[-1] + 1].to_numpy())

        def scaled_windows(positions):
            X, y, kept = make_windowed_arrays(df, feature_cols, target_col, positions)
            n, lb, nf = X.shape
            X_scaled = scaler.transform(X.reshape(-1, nf)).reshape(n, lb, nf).astype(np.float32)
            return X_scaled, y

        X_fit, y_fit = scaled_windows(fit_positions)
        X_val, y_val = scaled_windows(val_positions)
        n_features = len(feature_cols)

        model = LSTMRegressor(n_features=n_features)
        model = _train_sequence_model(model, X_fit, y_fit, X_val, y_val,
                                      max_epochs=FINAL_SEQ_MAX_EPOCHS,
                                      desc=f"lstm final refit [{suffix}]")
        torch.save(model.state_dict(), MODEL_DIR / f"lstm_final_{suffix}.pt")
        with open(MODEL_DIR / f"lstm_scaler_final_{suffix}.pkl", "wb") as f:
            pickle.dump({"scaler": scaler, "feature_cols": feature_cols,
                        "lookback": LOOKBACK_HOURS, "n_features": n_features}, f)
        log.info(f"Saved final lstm model + scaler -> {MODEL_DIR / f'lstm_final_{suffix}.pt'}")


if __name__ == "__main__":
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"{INPUT_PATH} not found — run Phase 3 first.")
    full_df = pd.read_parquet(INPUT_PATH)
    all_calibration_rows = []

    border_bar = tqdm(BORDERS, desc="Borders", unit="border")
    for zone_a, zone_b in border_bar:
        border_bar.set_postfix(border=f"{zone_a}-{zone_b}")
        target_col = f"spread_{zone_a}_{zone_b}"
        suffix = f"{zone_a}_{zone_b}"
        log.info(f"\n{'='*70}\nBorder {zone_a}-{zone_b}: target={target_col}\n{'='*70}")
        base_features = get_feature_columns(full_df)
        df, feature_cols = add_interaction_terms(full_df, base_features, zone_a, zone_b)
        log.info(f"[{target_col}] Using {len(feature_cols)} features (incl. interaction terms)")

        metrics_df, fold_preds, dm_df, calibration = run_walk_forward(df, feature_cols, target_col)
        metrics_df.to_csv(DATA_DIR / f"phase4_fold_metrics_{suffix}.csv", index=False)
        fold_preds.to_csv(DATA_DIR / f"phase4_test_predictions_{suffix}.csv")
        dm_df.to_csv(DATA_DIR / f"phase4_dm_tests_{suffix}.csv", index=False)

        if calibration:
            calibration_row = {"border": suffix, **calibration}
            all_calibration_rows.append(calibration_row)
            cal_df = pd.DataFrame([calibration_row]).drop(columns=["border"])
            style_table(
                cal_df, caption=f"Quantile calibration — {zone_a}-{zone_b} ([q10, q90] interval)",
                pct_cols=["empirical_coverage_below_q10", "target_coverage_below_q10",
                         "empirical_coverage_above_q90", "target_coverage_above_q90",
                         "empirical_coverage_within_interval", "target_coverage_within_interval"],
            )

        style_table(
            metrics_df.groupby("model")[["rmse", "mae", "directional_accuracy", "directional_accuracy_gt_1"]]
            .mean().reset_index().sort_values("rmse"),
            caption=f"Mean forecast accuracy across folds — {zone_a}-{zone_b}",
            highlight_min_cols=["rmse", "mae"], highlight_max_cols=["directional_accuracy_gt_1"],
        )

        with _timed(f"{suffix} final model refit"):
            fit_final_models(df, feature_cols, target_col, zone_a, zone_b)
    border_bar.close()

    if all_calibration_rows:
        pd.DataFrame(all_calibration_rows).to_csv(DATA_DIR / "phase4_calibration_all_borders.csv", index=False)


Borders:   0%|          | 0/3 [00:00<?, ?border/s]

2026-08-20 17:35:57,423 | INFO | 
Border DK1-DE_LU: target=spread_DK1_DE_LU
2026-08-20 17:35:57,428 | INFO | [spread_DK1_DE_LU] Using 57 features (incl. interaction terms)
2026-08-20 17:35:57,429 | INFO | [spread_DK1_DE_LU] Running 5 embargoed walk-forward folds (min train size 12788 rows, embargo 168h)


[spread_DK1_DE_LU] folds:   0%|          | 0/5 [00:00<?, ?fold/s]

[spread_DK1_DE_LU] Fold 0: train rows=12620, test rows=3836
  -> starting fold 0 elastic_net...
  -> fold 0 elastic_net done in 0.6s
  -> starting fold 0 lightgbm_point...
  -> fold 0 lightgbm_point done in 0.6s
  -> starting fold 0 lightgbm_quantiles...


2026-08-20 17:36:00,125 | WARNING | Dropping 168 position(s) with insufficient lookback history (need 168 prior rows) -- expected only at the very start of the dataset


...

  -> starting DK1_SE3 final model refit...


2026-08-21 15:28:02,564 | WARNING | Dropping 168 position(s) with insufficient lookback history (need 168 prior rows) -- expected only at the very start of the dataset


lstm final refit [DK1_SE3]:   0%|          | 0/25 [00:00<?, ?epoch/s]

2026-08-21 15:36:02,747 | INFO | Saved final lstm model + scaler -> models/lstm_final_DK1_SE3.pt


  -> DK1_SE3 final model refit done in 498.2s


# Model Accuracy, Trading Backtest and Robustness Tests

This section is split into 4 codeblocks:
1. Model accuracy summary for each border and model: RMSE, MAE, directional accuracy and Diebold-Mariano significance tests.
2. Trading backtest and signal generation
3. Roustness / stress tests
4. Serial dependence diagnostics

The backtest uses only held-out forecasts and information available before each delivery hour. The decision point is defined as day-ahead: realized market variables used for risk estimation are lagged by 24 hours, while ENTSO-E forecast products are treated as day-ahead forecast values for the delivery hour. Trade eligibility is based on forecasted edge, never on the realized spread. Volatility scaling and slippage are causal, threshold tuning is fold-by-fold, and all execution-cost assumptions are explicitly stylized rather than presented as actual institutional execution.


**Execution feasibility:** position size is additionally capped at a stylized fraction (`NTC_UTILIZATION_CAP`) of the day-ahead published NTC in the relevant direction, for borders where NTC data exists. This does not make the backtest a simulation of the real auction/allocation mechanics, but it does bound the notional traded by the interconnector's actual published capacity rather than leaving it unconstrained. A sequential cost waterfall (gross P&L, then BRP haircut, exchange fees, slippage, and margin financing subtracted one at a time) is reported alongside every strategy so the size of each stylized cost relative to the gross result is transparent rather than inferred.

In [5]:
"""Forecast Accuracy Summary
Reports point-forecast quality -- RMSE, MAE, directional accuracy, pinball
loss (for quantile models), quantile calibration, and Diebold-Mariano
significance tests, per border and per model."""

from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

DATA_DIR = Path("./data")
BORDERS = [("DK1", "DE_LU"), ("DK1", "DK2"), ("DK1", "SE3")]


def style_table(df: pd.DataFrame, caption: str = None, money_cols: list[str] = None,
                pct_cols: list[str] = None, highlight_max_cols: list[str] = None,
                highlight_min_cols: list[str] = None, decimals: int = 3):
    """Same styling helper as Phase 4 -- duplicated here (not imported) so
    this cell stays independently runnable, matching the rest of the notebook."""
    money_cols = money_cols or []
    pct_cols = pct_cols or []
    highlight_max_cols = highlight_max_cols or []
    highlight_min_cols = highlight_min_cols or []

    fmt = {}
    for c in df.columns:
        if c in money_cols:
            fmt[c] = "€{:,.0f}"
        elif c in pct_cols:
            fmt[c] = "{:.1%}"
        elif pd.api.types.is_float_dtype(df[c]):
            fmt[c] = f"{{:.{decimals}f}}"

    styler = df.style.format(fmt, na_rep="-").hide(axis="index")
    if caption:
        styler = styler.set_caption(caption)
    for c in highlight_max_cols:
        if c in df.columns:
            styler = styler.highlight_max(subset=[c], color="#c6efce")
    for c in highlight_min_cols:
        if c in df.columns:
            styler = styler.highlight_min(subset=[c], color="#ffc7ce")

    if HAS_IPYTHON:
        display(styler)
    else:
        print(f"\n{caption or ''}")
        print(df.to_string(index=False))
    return styler


def summarize_fold_metrics(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Mean +/- std of RMSE/MAE/directional accuracy/pinball loss across folds,
    per model. Pinball loss is only meaningful for the three quantile models
    (lightgbm_q10/q50/q90) -- it is NaN for every point-forecast model, which
    is expected, not a bug: RMSE and pinball loss are different scoring rules
    for different kinds of forecasts, and reporting one where it doesn't apply
    would be misleading rather than merely uninformative."""
    agg_kwargs = dict(
        rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"), mae_std=("mae", "std"),
        directional_accuracy_mean=("directional_accuracy", "mean"),
        directional_accuracy_gt_1_mean=("directional_accuracy_gt_1", "mean"),
        n_folds=("fold", "nunique"),
    )
    if "pinball_loss" in metrics_df.columns:
        agg_kwargs["pinball_loss_mean"] = ("pinball_loss", "mean")
        agg_kwargs["pinball_loss_std"] = ("pinball_loss", "std")
    agg = metrics_df.groupby("model").agg(**agg_kwargs).reset_index()
    return agg.sort_values("rmse_mean")


def format_metrics_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["rmse"] = out.apply(lambda r: f"{r.rmse_mean:.3f} ± {r.rmse_std:.3f}", axis=1)
    out["mae"] = out.apply(lambda r: f"{r.mae_mean:.3f} ± {r.mae_std:.3f}", axis=1)
    out["directional_accuracy"] = out["directional_accuracy_mean"]
    out["directional_accuracy_gt_1"] = out["directional_accuracy_gt_1_mean"]
    cols = ["model", "rmse", "mae", "directional_accuracy", "directional_accuracy_gt_1", "n_folds"]
    if "pinball_loss_mean" in out.columns:
        out["pinball_loss"] = out.apply(
            lambda r: f"{r.pinball_loss_mean:.3f} ± {r.pinball_loss_std:.3f}"
            if pd.notnull(r.pinball_loss_mean) else "n/a (point forecast)", axis=1
        )
        cols.insert(3, "pinball_loss")
    return out[cols]


def format_dm_table(dm_df: pd.DataFrame) -> pd.DataFrame:
    out = dm_df.copy()
    out["significant_5pct"] = dm_df["p_value"].map(
        lambda p: "yes" if pd.notnull(p) and p < 0.05 else ("no" if pd.notnull(p) else "N/A")
    )
    return out[["model_1", "model_2", "dm_stat", "p_value", "significant_5pct", "n", "horizon_lag"]]


all_metrics_summary = []
all_dm_results = []
all_calibration = []

calibration_path = DATA_DIR / "phase4_calibration_all_borders.csv"
calibration_all = pd.read_csv(calibration_path) if calibration_path.exists() else pd.DataFrame()

for zone_a, zone_b in BORDERS:
    suffix = f"{zone_a}_{zone_b}"
    metrics_path = DATA_DIR / f"phase4_fold_metrics_{suffix}.csv"
    dm_path = DATA_DIR / f"phase4_dm_tests_{suffix}.csv"

    if not metrics_path.exists():
        print(f"WARNING: {metrics_path} not found — skipping {zone_a}-{zone_b} "
              "(run Phase 4 for this border first)")
        continue

    metrics_df = pd.read_csv(metrics_path)
    metrics_summary = summarize_fold_metrics(metrics_df)
    metrics_summary_labeled = metrics_summary.copy()
    metrics_summary_labeled["border"] = suffix
    all_metrics_summary.append(metrics_summary_labeled)

    print(f"\n{'='*90}\nPHASE 5, PART 1 — FORECAST ACCURACY: {zone_a} <--> {zone_b}\n{'='*90}")
    style_table(
        format_metrics_table(metrics_summary),
        caption=f"RMSE / MAE / Directional Accuracy / Pinball Loss (mean ± std across folds) — {zone_a}-{zone_b}",
        pct_cols=["directional_accuracy", "directional_accuracy_gt_1"],
        highlight_max_cols=["directional_accuracy_gt_1"],
    )

    if dm_path.exists():
        dm_df = pd.read_csv(dm_path)
        dm_df_labeled = dm_df.copy()
        dm_df_labeled["border"] = suffix
        all_dm_results.append(dm_df_labeled)
        style_table(
            format_dm_table(dm_df),
            caption=f"Diebold-Mariano tests vs. persistence (positive = model_1 worse) — {zone_a}-{zone_b}",
        )
    else:
        print(f"\nWARNING: {dm_path} not found — no DM test results for this border.")

    if not calibration_all.empty and suffix in calibration_all["border"].values:
        cal_row = calibration_all[calibration_all["border"] == suffix].drop(columns=["border"])
        style_table(
            cal_row,
            caption=f"Quantile calibration ([q10, q90] interval) — {zone_a}-{zone_b}",
            pct_cols=[c for c in cal_row.columns if "coverage" in c],
        )

if all_metrics_summary:
    pd.concat(all_metrics_summary, ignore_index=True).to_csv(
        DATA_DIR / "phase5a_forecast_accuracy_summary_all_borders.csv", index=False
    )
if all_dm_results:
    pd.concat(all_dm_results, ignore_index=True).to_csv(
        DATA_DIR / "phase5a_dm_tests_all_borders.csv", index=False
    )



PHASE 5, PART 1 — FORECAST ACCURACY: DK1 <--> DE_LU


model,rmse,mae,pinball_loss,directional_accuracy,directional_accuracy_gt_1,n_folds
lightgbm_point,17.657 ± 4.080,9.417 ± 2.181,n/a (point forecast),39.3%,92.6%,5
elastic_net,18.545 ± 3.454,11.705 ± 2.104,n/a (point forecast),39.1%,92.4%,5
lstm,19.001 ± 5.129,9.479 ± 2.696,n/a (point forecast),35.9%,85.0%,5
lightgbm_q50,19.907 ± 5.229,9.317 ± 2.788,4.658 ± 1.394,48.8%,70.8%,5
lightgbm_q10,20.190 ± 1.817,13.114 ± 1.643,3.716 ± 1.617,37.3%,88.5%,5
lightgbm_q90,23.174 ± 6.269,10.524 ± 3.762,1.527 ± 0.297,51.4%,10.2%,5
persistence,25.701 ± 6.277,13.553 ± 4.039,n/a (point forecast),60.6%,53.2%,5
seasonal_naive,28.640 ± 6.534,15.852 ± 4.880,n/a (point forecast),52.7%,43.7%,5


model_1,model_2,dm_stat,p_value,significant_5pct,n,horizon_lag
elastic_net,persistence,-10.723,0.000,yes,19180,24
lightgbm_point,persistence,-10.882,0.000,yes,19180,24
lstm,persistence,-9.149,0.000,yes,19180,24
seasonal_naive,persistence,3.397,0.001,yes,19180,24


n_observations,empirical_coverage_below_q10,target_coverage_below_q10,empirical_coverage_above_q90,target_coverage_above_q90,empirical_coverage_within_interval,target_coverage_within_interval
19180,17.0%,10.0%,4.6%,10.0%,78.4%,80.0%



PHASE 5, PART 1 — FORECAST ACCURACY: DK1 <--> DK2


...

n_observations,empirical_coverage_below_q10,target_coverage_below_q10,empirical_coverage_above_q90,target_coverage_above_q90,empirical_coverage_within_interval,target_coverage_within_interval
19180,21.2%,10.0%,8.2%,10.0%,70.7%,80.0%



PHASE 5, PART 1 — FORECAST ACCURACY: DK1 <--> SE3


model,rmse,mae,pinball_loss,directional_accuracy,directional_accuracy_gt_1,n_folds
lightgbm_q50,29.547 ± 7.969,20.640 ± 6.759,10.320 ± 3.379,79.0%,90.3%,5
lightgbm_point,29.872 ± 8.613,21.034 ± 6.023,n/a (point forecast),79.3%,90.9%,5
elastic_net,31.715 ± 7.620,23.674 ± 6.520,n/a (point forecast),74.3%,85.0%,5
lstm,38.391 ± 11.192,27.138 ± 8.995,n/a (point forecast),76.8%,87.7%,5
persistence,39.927 ± 8.297,26.553 ± 6.140,n/a (point forecast),73.3%,79.4%,5
lightgbm_q90,40.861 ± 6.006,32.866 ± 6.363,4.992 ± 1.136,75.9%,87.2%,5
lightgbm_q10,42.107 ± 13.450,29.990 ± 11.251,4.210 ± 0.841,72.4%,82.1%,5
seasonal_naive,47.769 ± 10.153,33.502 ± 8.006,n/a (point forecast),68.4%,74.2%,5


model_1,model_2,dm_stat,p_value,significant_5pct,n,horizon_lag
elastic_net,persistence,-8.987,0.000,yes,19180,24
lightgbm_point,persistence,-9.101,0.000,yes,19180,24
lstm,persistence,-0.890,0.374,no,19180,24
seasonal_naive,persistence,5.722,0.000,yes,19180,24


n_observations,empirical_coverage_below_q10,target_coverage_below_q10,empirical_coverage_above_q90,target_coverage_above_q90,empirical_coverage_within_interval,target_coverage_within_interval
19180,14.9%,10.0%,12.5%,10.0%,72.6%,80.0%


In [6]:
"""Trading Backtest & Signal Generation The backtest uses held-out forecasts from Phase 4."""

import logging
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)


def style_table(df: pd.DataFrame, caption: str = None, money_cols: list[str] = None,
                pct_cols: list[str] = None, highlight_max_cols: list[str] = None,
                highlight_min_cols: list[str] = None, decimals: int = 3):
    """Styling helper duplicated across phases (not imported) so each cell
    stays independently runnable."""
    money_cols = money_cols or []
    pct_cols = pct_cols or []
    highlight_max_cols = highlight_max_cols or []
    highlight_min_cols = highlight_min_cols or []

    fmt = {}
    for c in df.columns:
        if c in money_cols:
            fmt[c] = "€{:,.0f}"
        elif c in pct_cols:
            fmt[c] = "{:.1%}"
        elif pd.api.types.is_float_dtype(df[c]):
            fmt[c] = f"{{:.{decimals}f}}"

    styler = df.style.format(fmt, na_rep="-").hide(axis="index")
    if caption:
        styler = styler.set_caption(caption)
    for c in highlight_max_cols:
        if c in df.columns:
            styler = styler.highlight_max(subset=[c], color="#c6efce")
    for c in highlight_min_cols:
        if c in df.columns:
            styler = styler.highlight_min(subset=[c], color="#ffc7ce")

    if HAS_IPYTHON:
        display(styler)
    else:
        print(f"\n{caption or ''}")
        print(df.to_string(index=False))
    return styler

DATA_DIR = Path("./data")
ZONES = {"DK1": "DK_1", "DE_LU": "DE_LU", "DK2": "DK_2", "SE3": "SE_3"}
BORDERS = [("DK1", "DE_LU"), ("DK1", "DK2"), ("DK1", "SE3")]

EXCHANGE_FEE_PER_LEG = 0.055
TRANSACTION_COST = EXCHANGE_FEE_PER_LEG * 2
THETA = 2.0
THETA_GRID = [0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0]
MIN_PREDICTED_EDGE_EUR = 2.0
HOURS_PER_YEAR = 24 * 365
MARGIN_Z = 2.33
NOTIONAL_MWH_PER_UNIT = 100.0
ANNUAL_FINANCING_RATE = 0.035
REALIZED_VOL_WINDOW_HOURS = 24 * 30
REALIZED_VOL_MIN_PERIODS = 24 * 7
REALIZED_VOL_LAG_HOURS = 24
BASE_SLIPPAGE_EUR_PER_MWH = 0.50
REFERENCE_VOL_EUR = 10.0
MIN_LOT_SIZE_MW = 1.0
BRP_IMBALANCE_HAIRCUT = 0.10
HAC_LAG_DAYS_GRID = [1, 3, 5, 7, 14, 30]


NTC_UTILIZATION_CAP = 0.10


def fixed_size_position(pred: pd.Series, theta: float = THETA) -> pd.Series:
    threshold = max(theta, MIN_PREDICTED_EDGE_EUR)
    pos = pd.Series(0.0, index=pred.index, dtype=float)
    pos[pred > threshold] = 1.0
    pos[pred < -threshold] = -1.0
    return pos


def volatility_target_scale(realized_vol: pd.Series, target_vol: float = REFERENCE_VOL_EUR) -> pd.Series:
    clean_vol = realized_vol.where(realized_vol > 0)
    scale = (target_vol / clean_vol).clip(lower=0.10, upper=1.0)
    return scale.fillna(0.0)


def enforce_minimum_lot_size(position_mwh: pd.Series, min_lot_mw: float = MIN_LOT_SIZE_MW) -> pd.Series:
    discretized = np.round(position_mwh / min_lot_mw) * min_lot_mw
    discretized[discretized.abs() < min_lot_mw] = 0.0
    return discretized


def apply_ntc_feasibility_cap(raw_mwh: pd.Series, ntc_forward: pd.Series | None,
                               ntc_reverse: pd.Series | None,
                               utilization_cap: float = NTC_UTILIZATION_CAP) -> pd.Series:
    """Cap the magnitude of a raw MWh position by a fraction of the day-ahead published NTC capacity in the relevant direction -- see NTC_UTILIZATION_CAP above for why."""
    if ntc_forward is None or ntc_reverse is None:
        return raw_mwh

    cap_fwd = (utilization_cap * ntc_forward.reindex(raw_mwh.index)).clip(lower=0).fillna(0.0)
    cap_rev = (utilization_cap * ntc_reverse.reindex(raw_mwh.index)).clip(lower=0).fillna(0.0)

    capped = raw_mwh.copy()
    long_mask = raw_mwh > 0
    short_mask = raw_mwh < 0
    capped[long_mask] = np.minimum(raw_mwh[long_mask], cap_fwd[long_mask])
    capped[short_mask] = np.maximum(raw_mwh[short_mask], -cap_rev[short_mask])
    return capped


def finalize_position_mwh(direction_units: pd.Series, vol_scale: pd.Series,
                           ntc_forward: pd.Series | None = None,
                           ntc_reverse: pd.Series | None = None,
                           min_lot_mw: float = MIN_LOT_SIZE_MW) -> pd.Series:
    """Single place where a unitless direction/confidence signal becomes a real MWh position: notional scaling, volatility targeting, the NTC capacity cap, and minimum-lot discretization, in that order."""
    idx = direction_units.index
    raw_mwh = direction_units * NOTIONAL_MWH_PER_UNIT * vol_scale.loc[idx]
    raw_mwh = apply_ntc_feasibility_cap(raw_mwh, ntc_forward, ntc_reverse)
    return enforce_minimum_lot_size(raw_mwh, min_lot_mw)


def confidence_weighted_position(q10: pd.Series, q50: pd.Series, q90: pd.Series,
                                 theta: float = THETA) -> pd.Series:
    direction = np.sign(q50)
    interval_width = (q90 - q10).clip(lower=1e-6)
    wrong_side_width = np.where(direction >= 0, (-q10).clip(lower=0), (q90).clip(lower=0))
    confidence = np.clip(1.0 - (wrong_side_width / interval_width), 0.0, 1.0)
    position = direction * confidence
    threshold = max(theta, MIN_PREDICTED_EDGE_EUR)
    position = np.where(q50.abs() > threshold, position, 0.0)
    return pd.Series(position, index=q50.index)


def compute_realized_volatility(full_spread: pd.Series,
                                window: int = REALIZED_VOL_WINDOW_HOURS,
                                min_periods: int = REALIZED_VOL_MIN_PERIODS,
                                information_lag_hours: int = REALIZED_VOL_LAG_HOURS) -> pd.Series:
    """Rolling realized spread volatility from the continuous target history."""
    return full_spread.sort_index().rolling(window=window, min_periods=min_periods).std().shift(information_lag_hours)


def compute_linear_dynamic_slippage(realized_vol: pd.Series,
                                    base_slippage: float = BASE_SLIPPAGE_EUR_PER_MWH,
                                    reference_vol: float = REFERENCE_VOL_EUR) -> pd.Series:
    clean_vol = realized_vol.where(realized_vol > 0)
    ratio = (clean_vol / reference_vol).clip(lower=0.5, upper=5.0)
    return (base_slippage * ratio).fillna(base_slippage)


def compute_margin_financing_cost(position_mwh: pd.Series, realized_vol: pd.Series,
                                  z: float = MARGIN_Z,
                                  annual_rate: float = ANNUAL_FINANCING_RATE,
                                  hours_per_year: int = HOURS_PER_YEAR) -> pd.Series:
    margin_eur = z * realized_vol * position_mwh.abs()
    financing_eur = margin_eur * annual_rate / hours_per_year
    return financing_eur.fillna(0.0)


def compute_pnl(position_mwh: pd.Series, actual_spread: pd.Series,
                dynamic_slippage_per_mwh: pd.Series,
                transaction_cost: float = TRANSACTION_COST,
                brp_haircut: float = BRP_IMBALANCE_HAIRCUT,
                margin_financing_cost: pd.Series | None = None) -> tuple[pd.Series, pd.Series]:
    effective_pos_mwh = position_mwh.fillna(0.0)
    gross_pnl_eur = effective_pos_mwh * actual_spread * (1.0 - brp_haircut)
    tx_costs_eur = transaction_cost * np.abs(effective_pos_mwh)
    market_impact_eur = dynamic_slippage_per_mwh * np.abs(effective_pos_mwh)
    pnl = gross_pnl_eur - tx_costs_eur - market_impact_eur
    if margin_financing_cost is not None:
        pnl = pnl - margin_financing_cost
    return pd.Series(pnl, index=actual_spread.index), pd.Series(market_impact_eur, index=actual_spread.index)


def sharpe_ratio(pnl: pd.Series, annual_days: int = 365, max_lag_days: int = 7) -> float:
    """HAC-adjusted annualized Sharpe based on daily P&L."""
    if pnl.empty:
        return 0.0
    daily = pnl.resample("D").sum().dropna()
    if len(daily) < max(20, max_lag_days + 2):
        return 0.0
    x = daily.to_numpy(dtype=float)
    mu = x.mean()
    centered = x - mu
    gamma0 = np.mean(centered ** 2)
    if gamma0 <= 0:
        return 0.0
    q = min(max_lag_days, len(x) - 1)
    long_run = gamma0
    for k in range(1, q + 1):
        weight = 1.0 - k / (q + 1.0)
        cov = np.mean(centered[k:] * centered[:-k])
        long_run += 2.0 * weight * cov
    return float(mu / np.sqrt(max(long_run, 1e-12)) * np.sqrt(annual_days))


def max_drawdown(cumulative_pnl: pd.Series) -> float:
    running_max = cumulative_pnl.cummax()
    return float((cumulative_pnl - running_max).min())


def win_rate(pnl: pd.Series, position_mwh: pd.Series) -> float:
    active = pnl[position_mwh != 0]
    return float((active > 0).mean()) if len(active) else float("nan")


def summarize_strategy(name: str, position_mwh: pd.Series, actual_spread: pd.Series,
                      dynamic_slippage_per_mwh: pd.Series,
                      margin_financing_cost: pd.Series | None = None,
                      brp_haircut: float = BRP_IMBALANCE_HAIRCUT) -> tuple[dict, pd.Series]:
    pnl, market_impact = compute_pnl(position_mwh, actual_spread, dynamic_slippage_per_mwh,
                                     margin_financing_cost=margin_financing_cost,
                                     brp_haircut=brp_haircut)
    cum_pnl = pnl.cumsum()

    pos = position_mwh.fillna(0.0)
    gross_no_costs = float((pos * actual_spread).sum())
    after_haircut = float((pos * actual_spread * (1.0 - brp_haircut)).sum())
    tx_total = float((TRANSACTION_COST * pos.abs()).sum())
    after_fees = after_haircut - tx_total
    slippage_total = float(market_impact.sum())
    after_slippage = after_fees - slippage_total
    margin_total = float(margin_financing_cost.sum()) if margin_financing_cost is not None else 0.0
    after_margin = after_slippage - margin_total

    summary = {
        "strategy": name,
        "cumulative_pnl_eur": float(cum_pnl.iloc[-1]) if len(cum_pnl) else 0.0,
        "hac_daily_sharpe": sharpe_ratio(pnl, max_lag_days=7),
        "max_drawdown_eur": max_drawdown(cum_pnl),
        "win_rate": win_rate(pnl, position_mwh),
        "active_hours": int((position_mwh != 0).sum()),
        "active_share": float((position_mwh != 0).mean()) if len(position_mwh) else float("nan"),
        "avg_abs_position": float((position_mwh / NOTIONAL_MWH_PER_UNIT).abs().mean()) if len(position_mwh) else float("nan"),
        "max_abs_position": float((position_mwh / NOTIONAL_MWH_PER_UNIT).abs().max()) if len(position_mwh) else float("nan"),
        "total_margin_cost_eur": margin_total,
        "total_slippage_cost_eur": slippage_total,
        "brp_haircut": brp_haircut,
        "waterfall_gross_pnl_no_costs_eur": gross_no_costs,
        "waterfall_after_brp_haircut_eur": after_haircut,
        "waterfall_after_exchange_fees_eur": after_fees,
        "waterfall_after_slippage_eur": after_slippage,
        "waterfall_after_margin_financing_eur": after_margin,
    }
    return summary, cum_pnl


def select_theta_from_past(pred_df: pd.DataFrame, model_col: str,
                           realized_vol: pd.Series, dynamic_slippage_per_mwh: pd.Series,
                           vol_scale: pd.Series,
                           ntc_forward: pd.Series | None = None,
                           ntc_reverse: pd.Series | None = None) -> float:
    if pred_df.empty:
        return THETA
    best_theta, best_score = THETA, -np.inf
    for theta in THETA_GRID:
        pos = fixed_size_position(pred_df[model_col], theta=theta)
        pos_mwh = finalize_position_mwh(pos, vol_scale.loc[pred_df.index], ntc_forward, ntc_reverse)
        m_cost = compute_margin_financing_cost(pos_mwh, realized_vol.loc[pred_df.index])
        pnl, _ = compute_pnl(pos_mwh, pred_df["y_true"], dynamic_slippage_per_mwh.loc[pred_df.index],
                             margin_financing_cost=m_cost)
        daily = pnl.resample("D").sum().dropna()
        score = float(daily.mean()) if len(daily) else -np.inf
        if score > best_score:
            best_score, best_theta = score, theta
    return float(best_theta)


def run_backtest(df: pd.DataFrame, historical_spread: pd.Series,
                 ntc_forward: pd.Series | None = None, ntc_reverse: pd.Series | None = None):
    actual = df["y_true"]
    historical_spread = historical_spread.sort_index()
    realized_vol = compute_realized_volatility(historical_spread)
    dynamic_slippage_per_mwh = compute_linear_dynamic_slippage(realized_vol)
    vol_scale = volatility_target_scale(realized_vol)

    non_model_cols = {"y_true", "fold"}
    point_model_cols = [c for c in df.columns if c not in non_model_cols and not c.startswith("lightgbm_q")]
    results, pnl_curves, theta_records = [], {}, []
    fold_values = sorted(df["fold"].dropna().unique())

    for col in point_model_cols:
        parts = []
        for fold_no in fold_values:
            fold_df = df[df["fold"] == fold_no]
            prior_df = df[df["fold"] < fold_no]
            theta = THETA if fold_no == fold_values[0] else select_theta_from_past(
                prior_df, col, realized_vol, dynamic_slippage_per_mwh, vol_scale, ntc_forward, ntc_reverse
            )
            theta_records.append({"model": col, "fold": int(fold_no), "theta_used": theta})
            pos = fixed_size_position(fold_df[col], theta=theta)
            parts.append(finalize_position_mwh(pos, vol_scale.loc[fold_df.index], ntc_forward, ntc_reverse))
        position = pd.concat(parts).sort_index()
        m_cost = compute_margin_financing_cost(position, realized_vol.loc[position.index])
        summary, cum_pnl = summarize_strategy(
            f"{col}_fold_tuned", position, actual.loc[position.index],
            dynamic_slippage_per_mwh.loc[position.index], margin_financing_cost=m_cost
        )
        results.append(summary)
        pnl_curves[f"{col}_fold_tuned"] = cum_pnl

    q_cols = {"lightgbm_q10", "lightgbm_q50", "lightgbm_q90"}
    if q_cols.issubset(df.columns):
        groups = {"lightgbm_confidence_weighted": []}
        for fold_no in fold_values:
            fold_df = df[df["fold"] == fold_no]; prior_df = df[df["fold"] < fold_no]
            theta = THETA if fold_no == fold_values[0] else select_theta_from_past(
                prior_df, "lightgbm_q50", realized_vol, dynamic_slippage_per_mwh, vol_scale, ntc_forward, ntc_reverse
            )
            q10, q50, q90 = fold_df["lightgbm_q10"], fold_df["lightgbm_q50"], fold_df["lightgbm_q90"]
            vs = vol_scale.loc[fold_df.index]
            groups["lightgbm_confidence_weighted"].append(finalize_position_mwh(
                confidence_weighted_position(q10, q50, q90, theta=theta), vs, ntc_forward, ntc_reverse))
        for label, parts in groups.items():
            position = pd.concat(parts).sort_index()
            m_cost = compute_margin_financing_cost(position, realized_vol.loc[position.index])
            summary, cum_pnl = summarize_strategy(
                label, position, actual.loc[position.index], dynamic_slippage_per_mwh.loc[position.index], margin_financing_cost=m_cost
            )
            results.append(summary); pnl_curves[label] = cum_pnl

    results_df = pd.DataFrame(results)
    return results_df, pd.DataFrame(pnl_curves), realized_vol, dynamic_slippage_per_mwh, vol_scale, pd.DataFrame(theta_records)


def run_hac_sensitivity(results_pnl: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for strategy in results_pnl.columns:
        pnl = results_pnl[strategy]
        for lag in HAC_LAG_DAYS_GRID:
            rows.append({"strategy": strategy, "hac_lag_days": lag,
                         "hac_daily_sharpe": sharpe_ratio(pnl, max_lag_days=lag)})
    return pd.DataFrame(rows)


if __name__ == "__main__":
    historical_path = DATA_DIR / "clean_hourly_dataset.parquet"
    if not historical_path.exists():
        raise FileNotFoundError(f"{historical_path} not found — run Phase 2 first.")
    historical_df = pd.read_parquet(historical_path)

    print("=" * 80)
    print("READING NOTE: cumulative PnL figures below assume a "
          f"{NOTIONAL_MWH_PER_UNIT:.0f} MWh/unit position cleared at the realized "
          "day-ahead price. This is a directional bet on two separately-forecast "
          "prices, NOT risk-free arbitrage, and is NOT a claim that this size was "
          "actually executable in the real auction. See the module docstring and "
          "the cost waterfall / NTC-capacity-capped positions below for the two "
          "concrete steps taken to narrow that gap.")
    print("=" * 80)

    for zone_a, zone_b in BORDERS:
        suffix = f"{zone_a}_{zone_b}"
        input_path = DATA_DIR / f"phase4_test_predictions_{suffix}.csv"
        if not input_path.exists():
            log.warning(f"{input_path} not found — skipping {zone_a}-{zone_b}.")
            continue


        df = pd.read_csv(input_path, index_col=0)
        df.index = pd.to_datetime(df.index, utc=True).tz_convert("Europe/Copenhagen")
        target_col = f"spread_{zone_a}_{zone_b}"
        if target_col not in historical_df.columns:
            raise KeyError(f"{target_col} not found in clean historical dataset.")
        historical_spread = historical_df[target_col].dropna()


        code_a, code_b = ZONES[zone_a], ZONES[zone_b]
        ntc_fwd_col, ntc_rev_col = f"ntc_{code_a}_to_{code_b}", f"ntc_{code_b}_to_{code_a}"
        if ntc_fwd_col in historical_df.columns and ntc_rev_col in historical_df.columns\
                and historical_df[ntc_fwd_col].notna().any():
            ntc_forward = historical_df[ntc_fwd_col].sort_index()
            ntc_reverse = historical_df[ntc_rev_col].sort_index()
            log.info(f"{zone_a}-{zone_b}: applying {NTC_UTILIZATION_CAP:.0%} NTC-capacity "
                     "feasibility cap to position sizing")
        else:
            ntc_forward = ntc_reverse = None
            log.info(f"{zone_a}-{zone_b}: no NTC data available — positions NOT capacity-capped "
                     "(read as an optimistic upper bound relative to the NTC-capped borders)")

        results_df, pnl_curves_df, realized_vol, slippage, vol_scale, theta_used = run_backtest(
            df, historical_spread, ntc_forward, ntc_reverse
        )
        results_df.to_csv(DATA_DIR / f"phase5_backtest_results_{suffix}.csv", index=False)
        pnl_curves_df.to_csv(DATA_DIR / f"phase5_pnl_curves_{suffix}.csv")
        theta_used.to_csv(DATA_DIR / f"phase5_theta_used_{suffix}.csv", index=False)

        hac_df = run_hac_sensitivity(pnl_curves_df)
        hac_df.to_csv(DATA_DIR / f"phase5_hac_sensitivity_{suffix}.csv", index=False)

        print("\n" + "=" * 80)
        print(f"  BORDER: {zone_a} <--> {zone_b} (Stylized Execution-Cost and BRP-Risk Backtest)")
        print("=" * 80)
        style_table(
            results_df, caption=f"Backtest summary — {zone_a}-{zone_b}",
            money_cols=["cumulative_pnl_eur", "max_drawdown_eur", "total_margin_cost_eur",
                       "total_slippage_cost_eur"],
            pct_cols=["win_rate", "active_share"],
            highlight_max_cols=["cumulative_pnl_eur", "hac_daily_sharpe"],
            highlight_min_cols=["max_drawdown_eur"],
        )
        waterfall_cols = ["strategy", "waterfall_gross_pnl_no_costs_eur", "waterfall_after_brp_haircut_eur",
                          "waterfall_after_exchange_fees_eur", "waterfall_after_slippage_eur",
                          "waterfall_after_margin_financing_eur"]
        style_table(
            results_df[waterfall_cols], caption=f"Cost waterfall (gross -> net) — {zone_a}-{zone_b}",
            money_cols=waterfall_cols[1:],
        )
        print("\nHAC LAG SENSITIVITY — all strategies saved to CSV.")

READING NOTE: cumulative PnL figures below assume a 100 MWh/unit position cleared at the realized day-ahead price. This is a directional bet on two separately-forecast prices, NOT risk-free arbitrage, and is NOT a claim that this size was actually executable in the real auction. See the module docstring and the cost waterfall / NTC-capacity-capped positions below for the two concrete steps taken to narrow that gap.


2026-08-21 15:36:03,375 | INFO | DK1-DE_LU: applying 10% NTC-capacity feasibility cap to position sizing



  BORDER: DK1 <--> DE_LU (Stylized Execution-Cost and BRP-Risk Backtest)


strategy,cumulative_pnl_eur,hac_daily_sharpe,max_drawdown_eur,win_rate,active_hours,active_share,avg_abs_position,max_abs_position,total_margin_cost_eur,total_slippage_cost_eur,brp_haircut,waterfall_gross_pnl_no_costs_eur,waterfall_after_brp_haircut_eur,waterfall_after_exchange_fees_eur,waterfall_after_slippage_eur,waterfall_after_margin_financing_eur
persistence_fold_tuned,"€4,323,182",6.743,"€-44,887",51.1%,6759,35.2%,0.195,1.000,€62,"€330,698",0.100,5216836.230,4695152.607,4653940.997,4323243.342,4323181.770
seasonal_naive_fold_tuned,"€2,900,959",5.488,"€-71,938",40.0%,6838,35.7%,0.193,1.000,€63,"€335,958",0.100,3641977.400,3277779.660,3236980.000,2901021.685,2900959.134
elastic_net_fold_tuned,"€7,652,063",7.714,"€-21,628",43.2%,14397,75.1%,0.432,1.000,€131,"€704,464",0.100,9386416.790,8447775.111,8356658.041,7652194.406,7652063.244
lightgbm_point_fold_tuned,"€7,621,366",7.744,"€-17,807",52.8%,11010,57.4%,0.335,1.000,€100,"€537,492",0.100,9144036.620,8229632.958,8158957.738,7621465.708,7621365.633
lstm_fold_tuned,"€6,604,499",6.860,"€-16,229",58.8%,7564,39.4%,0.222,1.000,€69,"€370,623",0.100,7802172.110,7021954.899,6975191.259,6604568.330,6604499.325
lightgbm_confidence_weighted,"€5,241,830",5.753,"€-45,638",64.8%,5761,30.0%,0.173,1.000,€50,"€265,970",0.100,6160407.660,5544366.894,5507849.424,5241879.498,5241829.978


strategy,waterfall_gross_pnl_no_costs_eur,waterfall_after_brp_haircut_eur,waterfall_after_exchange_fees_eur,waterfall_after_slippage_eur,waterfall_after_margin_financing_eur
persistence_fold_tuned,"€5,216,836","€4,695,153","€4,653,941","€4,323,243","€4,323,182"
seasonal_naive_fold_tuned,"€3,641,977","€3,277,780","€3,236,980","€2,901,022","€2,900,959"
elastic_net_fold_tuned,"€9,386,417","€8,447,775","€8,356,658","€7,652,194","€7,652,063"
lightgbm_point_fold_tuned,"€9,144,037","€8,229,633","€8,158,958","€7,621,466","€7,621,366"
lstm_fold_tuned,"€7,802,172","€7,021,955","€6,975,191","€6,604,568","€6,604,499"
lightgbm_confidence_weighted,"€6,160,408","€5,544,367","€5,507,849","€5,241,879","€5,241,830"


...

2026-08-21 15:36:08,500 | INFO | DK1-SE3: no NTC data available — positions NOT capacity-capped (read as an optimistic upper bound relative to the NTC-capped borders)



HAC LAG SENSITIVITY — all strategies saved to CSV.

  BORDER: DK1 <--> SE3 (Stylized Execution-Cost and BRP-Risk Backtest)


strategy,cumulative_pnl_eur,hac_daily_sharpe,max_drawdown_eur,win_rate,active_hours,active_share,avg_abs_position,max_abs_position,total_margin_cost_eur,total_slippage_cost_eur,brp_haircut,waterfall_gross_pnl_no_costs_eur,waterfall_after_brp_haircut_eur,waterfall_after_exchange_fees_eur,waterfall_after_slippage_eur,waterfall_after_margin_financing_eur
persistence_fold_tuned,"€13,589,913",10.951,"€-32,332",76.7%,15137,78.9%,0.226,0.670,€141,"€751,631",0.100,15988160.780,14389344.702,14341684.562,13590053.928,13589913.040
seasonal_naive_fold_tuned,"€11,887,088",9.395,"€-110,669",71.0%,15170,79.1%,0.227,0.670,€141,"€753,242",0.100,14098133.020,12688319.718,12640470.488,11887228.938,11887087.741
elastic_net_fold_tuned,"€14,959,380",11.453,"€-98,578",72.4%,17555,91.5%,0.264,0.670,€163,"€870,614",0.100,17650842.810,15885758.529,15830157.159,14959543.495,14959380.103
lightgbm_point_fold_tuned,"€15,913,515",13.161,"€-24,491",75.3%,18386,95.9%,0.281,0.670,€171,"€911,807",0.100,18760891.500,16884802.350,16825492.550,15913686.018,15913514.896
lstm_fold_tuned,"€14,970,669",11.806,"€-120,781",76.2%,16730,87.2%,0.254,0.670,€156,"€829,664",0.100,17615628.440,15854065.596,15800487.896,14970824.371,14970668.660
lightgbm_confidence_weighted,"€15,621,230",12.574,"€-38,292",76.5%,17650,92.0%,0.249,0.670,€153,"€817,068",0.100,18323285.190,16490956.671,16438451.691,15621383.664,15621230.290


strategy,waterfall_gross_pnl_no_costs_eur,waterfall_after_brp_haircut_eur,waterfall_after_exchange_fees_eur,waterfall_after_slippage_eur,waterfall_after_margin_financing_eur
persistence_fold_tuned,"€15,988,161","€14,389,345","€14,341,685","€13,590,054","€13,589,913"
seasonal_naive_fold_tuned,"€14,098,133","€12,688,320","€12,640,470","€11,887,229","€11,887,088"
elastic_net_fold_tuned,"€17,650,843","€15,885,759","€15,830,157","€14,959,543","€14,959,380"
lightgbm_point_fold_tuned,"€18,760,892","€16,884,802","€16,825,493","€15,913,686","€15,913,515"
lstm_fold_tuned,"€17,615,628","€15,854,066","€15,800,488","€14,970,824","€14,970,669"
lightgbm_confidence_weighted,"€18,323,285","€16,490,957","€16,438,452","€15,621,384","€15,621,230"



HAC LAG SENSITIVITY — all strategies saved to CSV.


In [7]:
"""Conservative Execution Robustness / Stress Tests PURPOSE ------- Keep the existing Phase 5 backtest as the BASELINE specification."""

from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False


def style_table(df: pd.DataFrame, caption: str = None, money_cols: list[str] = None,
                pct_cols: list[str] = None, highlight_max_cols: list[str] = None,
                highlight_min_cols: list[str] = None, decimals: int = 3):
    """Styling helper duplicated across phases (not imported) so each cell
    stays independently runnable."""
    money_cols = money_cols or []
    pct_cols = pct_cols or []
    highlight_max_cols = highlight_max_cols or []
    highlight_min_cols = highlight_min_cols or []

    fmt = {}
    for c in df.columns:
        if c in money_cols:
            fmt[c] = "€{:,.0f}"
        elif c in pct_cols:
            fmt[c] = "{:.1%}"
        elif pd.api.types.is_float_dtype(df[c]):
            fmt[c] = f"{{:.{decimals}f}}"

    styler = df.style.format(fmt, na_rep="-").hide(axis="index")
    if caption:
        styler = styler.set_caption(caption)
    for c in highlight_max_cols:
        if c in df.columns:
            styler = styler.highlight_max(subset=[c], color="#c6efce")
    for c in highlight_min_cols:
        if c in df.columns:
            styler = styler.highlight_min(subset=[c], color="#ffc7ce")

    if HAS_IPYTHON:
        display(styler)
    else:
        print(f"\n{caption or ''}")
        print(df.to_string(index=False))
    return styler


# -----------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------

DATA_DIR = Path("./data")

ZONES = {
    "DK1": "DK_1",
    "DE_LU": "DE_LU",
    "DK2": "DK_2",
    "SE3": "SE_3",
}

BORDERS = [
    ("DK1", "DE_LU"),
    ("DK1", "DK2"),
    ("DK1", "SE3"),
]


# -----------------------------------------------------------------------
# CONSERVATIVE EXECUTION SCENARIOS
# -----------------------------------------------------------------------


# -----------------------------------------------------------------------

STRESS_SCENARIOS = {

    "moderate_stress": {
        "transaction_cost_eur_per_mwh": 0.16,
        "slippage_base_eur_per_mwh": 1.00,
        "brp_haircut": 0.20,
        "ntc_utilization_cap": 0.075,
        "no_ntc_hard_cap_mwh": 75.0,
    },

    "conservative_stress": {
        "transaction_cost_eur_per_mwh": 0.22,
        "slippage_base_eur_per_mwh": 1.50,
        "brp_haircut": 0.30,
        "ntc_utilization_cap": 0.05,
        "no_ntc_hard_cap_mwh": 50.0,
    },

    "severe_stress": {
        "transaction_cost_eur_per_mwh": 0.30,
        "slippage_base_eur_per_mwh": 2.00,
        "brp_haircut": 0.30,
        "ntc_utilization_cap": 0.025,
        "no_ntc_hard_cap_mwh": 25.0,
    },
}


# -----------------------------------------------------------------------
# KEEP RISK SETTINGS FROM BASELINE PHASE 5
# -----------------------------------------------------------------------


# -----------------------------------------------------------------------

STRESS_REFERENCE_VOL_EUR = REFERENCE_VOL_EUR
STRESS_NOTIONAL_MWH_PER_UNIT = NOTIONAL_MWH_PER_UNIT
STRESS_MIN_LOT_SIZE_MW = MIN_LOT_SIZE_MW
STRESS_FINANCING_RATE = ANNUAL_FINANCING_RATE
STRESS_MARGIN_Z = MARGIN_Z


# -----------------------------------------------------------------------
# SLIPPAGE
# -----------------------------------------------------------------------

def stress_dynamic_slippage(
    realized_vol: pd.Series,
    base_slippage: float,
) -> pd.Series:
    """Dynamic slippage under the stress scenario."""

    clean_vol = realized_vol.where(realized_vol > 0)

    ratio = (
        clean_vol / STRESS_REFERENCE_VOL_EUR
    ).clip(lower=0.5, upper=5.0)

    return (
        base_slippage * ratio
    ).fillna(base_slippage)


# -----------------------------------------------------------------------
# CAPACITY / EXECUTION FEASIBILITY
# -----------------------------------------------------------------------

def stress_position_cap(
    raw_mwh: pd.Series,
    ntc_forward: pd.Series | None,
    ntc_reverse: pd.Series | None,
    ntc_utilization_cap: float,
    no_ntc_hard_cap_mwh: float,
) -> pd.Series:
    """Apply the scenario's execution-capacity restriction."""

    capped = raw_mwh.copy()

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    if ntc_forward is not None and ntc_reverse is not None:

        cap_fwd = (
            ntc_utilization_cap
            * ntc_forward.reindex(raw_mwh.index)
        ).clip(lower=0).fillna(0.0)

        cap_rev = (
            ntc_utilization_cap
            * ntc_reverse.reindex(raw_mwh.index)
        ).clip(lower=0).fillna(0.0)

        long_mask = raw_mwh > 0
        short_mask = raw_mwh < 0

        capped[long_mask] = np.minimum(
            raw_mwh[long_mask],
            cap_fwd[long_mask],
        )

        capped[short_mask] = np.maximum(
            raw_mwh[short_mask],
            -cap_rev[short_mask],
        )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    else:

        capped = capped.clip(
            -no_ntc_hard_cap_mwh,
            no_ntc_hard_cap_mwh,
        )

    return capped


# -----------------------------------------------------------------------
# POSITION CONSTRUCTION
# -----------------------------------------------------------------------

def finalize_stress_position(
    direction_units: pd.Series,
    vol_scale: pd.Series,
    scenario: dict,
    ntc_forward: pd.Series | None,
    ntc_reverse: pd.Series | None,
) -> pd.Series:

    raw_mwh = (
        direction_units
        * STRESS_NOTIONAL_MWH_PER_UNIT
        * vol_scale.loc[direction_units.index]
    )

    raw_mwh = stress_position_cap(
        raw_mwh=raw_mwh,
        ntc_forward=ntc_forward,
        ntc_reverse=ntc_reverse,
        ntc_utilization_cap=scenario["ntc_utilization_cap"],
        no_ntc_hard_cap_mwh=scenario["no_ntc_hard_cap_mwh"],
    )


    result = (
        np.round(
            raw_mwh / STRESS_MIN_LOT_SIZE_MW
        )
        * STRESS_MIN_LOT_SIZE_MW
    )

    result[result.abs() < STRESS_MIN_LOT_SIZE_MW] = 0.0

    return result


# -----------------------------------------------------------------------
# MARGIN FINANCING
# -----------------------------------------------------------------------

def stress_margin_financing(
    position_mwh: pd.Series,
    realized_vol: pd.Series,
) -> pd.Series:

    margin_eur = (
        STRESS_MARGIN_Z
        * realized_vol.loc[position_mwh.index]
        * position_mwh.abs()
    )

    financing = (
        margin_eur
        * STRESS_FINANCING_RATE
        / HOURS_PER_YEAR
    )

    return financing.fillna(0.0)


# -----------------------------------------------------------------------
# P&L
# -----------------------------------------------------------------------

def stress_pnl(
    position_mwh: pd.Series,
    actual_spread: pd.Series,
    dynamic_slippage_per_mwh: pd.Series,
    scenario: dict,
    margin_financing_cost: pd.Series,
) -> tuple[pd.Series, pd.Series]:

    pos = position_mwh.fillna(0.0)

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    gross_after_brp = (
        pos
        * actual_spread
        * (1.0 - scenario["brp_haircut"])
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    transaction_fees = (
        scenario["transaction_cost_eur_per_mwh"]
        * pos.abs()
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    slippage = (
        dynamic_slippage_per_mwh.loc[pos.index]
        * pos.abs()
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    pnl = (
        gross_after_brp
        - transaction_fees
        - slippage
        - margin_financing_cost.loc[pos.index]
    )

    return pnl, slippage


# -----------------------------------------------------------------------
# SUMMARY STATISTICS
# -----------------------------------------------------------------------

def stress_summary(
    strategy_name: str,
    position_mwh: pd.Series,
    actual_spread: pd.Series,
    dynamic_slippage_per_mwh: pd.Series,
    scenario: dict,
    margin_financing_cost: pd.Series,
) -> tuple[dict, pd.Series]:

    pnl, market_impact = stress_pnl(
        position_mwh=position_mwh,
        actual_spread=actual_spread,
        dynamic_slippage_per_mwh=dynamic_slippage_per_mwh,
        scenario=scenario,
        margin_financing_cost=margin_financing_cost,
    )

    cumulative = pnl.cumsum()

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    daily = pnl.resample("D").sum().dropna()

    if (
        len(daily) >= 20
        and daily.std(ddof=0) > 0
    ):
        naive_daily_sharpe = float(
            daily.mean()
            / daily.std(ddof=0)
            * np.sqrt(365)
        )
    else:
        naive_daily_sharpe = 0.0

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    hac_7 = sharpe_ratio(
        pnl,
        max_lag_days=7,
    )

    hac_30 = sharpe_ratio(
        pnl,
        max_lag_days=30,
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    pos = position_mwh.fillna(0.0)

    gross_no_costs = float(
        (pos * actual_spread).sum()
    )

    after_brp = float(
        (
            pos
            * actual_spread
            * (1.0 - scenario["brp_haircut"])
        ).sum()
    )

    exchange_fees = float(
        (
            scenario["transaction_cost_eur_per_mwh"]
            * pos.abs()
        ).sum()
    )

    after_exchange_fees = (
        after_brp
        - exchange_fees
    )

    slippage_total = float(
        market_impact.sum()
    )

    after_slippage = (
        after_exchange_fees
        - slippage_total
    )

    margin_total = float(
        margin_financing_cost.sum()
    )

    after_margin = (
        after_slippage
        - margin_total
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    running_max = cumulative.cummax()

    max_dd = (
        float(
            (cumulative - running_max).min()
        )
        if len(cumulative)
        else 0.0
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    active = position_mwh != 0

    if active.any():
        win_rate_value = float(
            (pnl[active] > 0).mean()
        )
    else:
        win_rate_value = np.nan

    summary = {

        "strategy": strategy_name,

        "cumulative_pnl_eur": (
            float(cumulative.iloc[-1])
            if len(cumulative)
            else 0.0
        ),

        "naive_daily_sharpe": naive_daily_sharpe,

        "hac_daily_sharpe_7d": hac_7,

        "hac_daily_sharpe_30d": hac_30,

        "hac_30d_minus_7d": (
            hac_30 - hac_7
        ),

        "max_drawdown_eur": max_dd,

        "win_rate": win_rate_value,

        "active_hours": int(
            active.sum()
        ),

        "active_share": float(
            active.mean()
        ) if len(active) else np.nan,

        "avg_abs_position_units": float(
            (
                position_mwh
                / STRESS_NOTIONAL_MWH_PER_UNIT
            ).abs().mean()
        ),

        "max_abs_position_units": float(
            (
                position_mwh
                / STRESS_NOTIONAL_MWH_PER_UNIT
            ).abs().max()
        ),

        "total_margin_cost_eur": margin_total,

        "total_slippage_cost_eur": slippage_total,

        "brp_haircut": scenario["brp_haircut"],

        "transaction_cost_eur_per_mwh": (
            scenario["transaction_cost_eur_per_mwh"]
        ),

        "slippage_base_eur_per_mwh": (
            scenario["slippage_base_eur_per_mwh"]
        ),

        "ntc_utilization_cap": (
            scenario["ntc_utilization_cap"]
        ),

        "no_ntc_hard_cap_mwh": (
            scenario["no_ntc_hard_cap_mwh"]
        ),

        "gross_pnl_no_costs_eur": gross_no_costs,

        "after_brp_haircut_eur": after_brp,

        "after_exchange_fees_eur": after_exchange_fees,

        "after_slippage_eur": after_slippage,

        "after_margin_financing_eur": after_margin,
    }

    return summary, cumulative


# -----------------------------------------------------------------------
# FOLD-SAFE THRESHOLD SELECTION
# -----------------------------------------------------------------------

def select_theta_from_past_stress(
    pred_df: pd.DataFrame,
    model_col: str,
    realized_vol: pd.Series,
    dynamic_slippage_per_mwh: pd.Series,
    vol_scale: pd.Series,
    ntc_forward: pd.Series | None,
    ntc_reverse: pd.Series | None,
    scenario: dict,
) -> float:
    """Select theta using ONLY previous folds."""

    if pred_df.empty:
        return THETA

    best_theta = THETA
    best_score = -np.inf

    for theta in THETA_GRID:

        direction = fixed_size_position(
            pred_df[model_col],
            theta=theta,
        )

        position_mwh = finalize_stress_position(
            direction_units=direction,
            vol_scale=vol_scale,
            scenario=scenario,
            ntc_forward=ntc_forward,
            ntc_reverse=ntc_reverse,
        )

        margin = stress_margin_financing(
            position_mwh,
            realized_vol,
        )

        pnl, _ = stress_pnl(
            position_mwh=position_mwh,
            actual_spread=pred_df["y_true"],
            dynamic_slippage_per_mwh=dynamic_slippage_per_mwh,
            scenario=scenario,
            margin_financing_cost=margin,
        )

        daily = (
            pnl
            .resample("D")
            .sum()
            .dropna()
        )

        score = (
            float(daily.mean())
            if len(daily)
            else -np.inf
        )

        if score > best_score:
            best_score = score
            best_theta = theta

    return float(best_theta)


# -----------------------------------------------------------------------
# FULL STRESS BACKTEST
# -----------------------------------------------------------------------

def run_stress_backtest(
    pred_df: pd.DataFrame,
    historical_spread: pd.Series,
    scenario: dict,
    ntc_forward: pd.Series | None,
    ntc_reverse: pd.Series | None,
):

    actual = pred_df["y_true"]

    historical_spread = (
        historical_spread
        .sort_index()
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    realized_vol = compute_realized_volatility(
        historical_spread
    )

    dynamic_slippage = stress_dynamic_slippage(
        realized_vol=realized_vol,
        base_slippage=scenario[
            "slippage_base_eur_per_mwh"
        ],
    )

    vol_scale = volatility_target_scale(
        realized_vol
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    non_model_cols = {
        "y_true",
        "fold",
    }

    point_model_cols = [
        c
        for c in pred_df.columns
        if c not in non_model_cols
        and not c.startswith("lightgbm_q")
    ]

    fold_values = sorted(
        pred_df["fold"]
        .dropna()
        .unique()
    )

    results = []
    pnl_curves = {}
    theta_records = []

    for col in point_model_cols:

        position_parts = []

        for fold_no in fold_values:

            fold_df = pred_df[
                pred_df["fold"] == fold_no
            ]

            prior_df = pred_df[
                pred_df["fold"] < fold_no
            ]

            if fold_no == fold_values[0]:

                theta = THETA

            else:

                theta = select_theta_from_past_stress(
                    pred_df=prior_df,
                    model_col=col,
                    realized_vol=realized_vol,
                    dynamic_slippage_per_mwh=dynamic_slippage,
                    vol_scale=vol_scale,
                    ntc_forward=ntc_forward,
                    ntc_reverse=ntc_reverse,
                    scenario=scenario,
                )

            theta_records.append({
                "model": col,
                "fold": int(fold_no),
                "theta_used": theta,
            })

            direction = fixed_size_position(
                fold_df[col],
                theta=theta,
            )

            position_parts.append(
                finalize_stress_position(
                    direction_units=direction,
                    vol_scale=vol_scale.loc[
                        fold_df.index
                    ],
                    scenario=scenario,
                    ntc_forward=ntc_forward,
                    ntc_reverse=ntc_reverse,
                )
            )

        position = (
            pd.concat(position_parts)
            .sort_index()
        )

        margin = stress_margin_financing(
            position,
            realized_vol,
        )

        summary, cumulative = stress_summary(
            strategy_name=f"{col}_stress",
            position_mwh=position,
            actual_spread=actual.loc[
                position.index
            ],
            dynamic_slippage_per_mwh=dynamic_slippage,
            scenario=scenario,
            margin_financing_cost=margin,
        )

        results.append(summary)

        pnl_curves[
            summary["strategy"]
        ] = cumulative

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    q_cols = {
        "lightgbm_q10",
        "lightgbm_q50",
        "lightgbm_q90",
    }

    if q_cols.issubset(pred_df.columns):

        groups = {
            "lightgbm_confidence_weighted_stress": [],
        }

        for fold_no in fold_values:

            fold_df = pred_df[
                pred_df["fold"] == fold_no
            ]

            prior_df = pred_df[
                pred_df["fold"] < fold_no
            ]

            if fold_no == fold_values[0]:

                theta = THETA

            else:

                theta = select_theta_from_past_stress(
                    pred_df=prior_df,
                    model_col="lightgbm_q50",
                    realized_vol=realized_vol,
                    dynamic_slippage_per_mwh=dynamic_slippage,
                    vol_scale=vol_scale,
                    ntc_forward=ntc_forward,
                    ntc_reverse=ntc_reverse,
                    scenario=scenario,
                )

            q10 = fold_df["lightgbm_q10"]
            q50 = fold_df["lightgbm_q50"]
            q90 = fold_df["lightgbm_q90"]

            vs = vol_scale.loc[
                fold_df.index
            ]

            confidence_units = (
                confidence_weighted_position(
                    q10=q10,
                    q50=q50,
                    q90=q90,
                    theta=theta,
                )
            )

            groups[
                "lightgbm_confidence_weighted_stress"
            ].append(
                finalize_stress_position(
                    direction_units=confidence_units,
                    vol_scale=vs,
                    scenario=scenario,
                    ntc_forward=ntc_forward,
                    ntc_reverse=ntc_reverse,
                )
            )

        for label, parts in groups.items():

            position = (
                pd.concat(parts)
                .sort_index()
            )

            margin = stress_margin_financing(
                position,
                realized_vol,
            )

            summary, cumulative = stress_summary(
                strategy_name=label,
                position_mwh=position,
                actual_spread=actual.loc[
                    position.index
                ],
                dynamic_slippage_per_mwh=dynamic_slippage,
                scenario=scenario,
                margin_financing_cost=margin,
            )

            results.append(summary)

            pnl_curves[
                label
            ] = cumulative

    return (
        pd.DataFrame(results),
        pd.DataFrame(pnl_curves),
        pd.DataFrame(theta_records),
    )


# -----------------------------------------------------------------------
# HAC SENSITIVITY
# -----------------------------------------------------------------------

def run_stress_hac_sensitivity(
    pnl_df: pd.DataFrame,
) -> pd.DataFrame:

    rows = []

    for strategy in pnl_df.columns:

        pnl = pnl_df[strategy]

        for lag in HAC_LAG_DAYS_GRID:

            rows.append({
                "strategy": strategy,
                "hac_lag_days": lag,
                "hac_daily_sharpe": sharpe_ratio(
                    pnl,
                    max_lag_days=lag,
                ),
            })

    return pd.DataFrame(rows)


# -----------------------------------------------------------------------
# RUN ALL BORDERS / SCENARIOS
# -----------------------------------------------------------------------

historical_path = (
    DATA_DIR
    / "clean_hourly_dataset.parquet"
)

if not historical_path.exists():

    raise FileNotFoundError(
        f"{historical_path} not found — run Phase 2 first."
    )

historical_df = pd.read_parquet(
    historical_path
)

all_stress_results = []


for zone_a, zone_b in BORDERS:

    suffix = f"{zone_a}_{zone_b}"

    input_path = (
        DATA_DIR
        / f"phase4_test_predictions_{suffix}.csv"
    )

    if not input_path.exists():

        print(
            f"WARNING: {input_path} not found — "
            f"skipping {zone_a}-{zone_b}."
        )

        continue

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    pred_df = pd.read_csv(
        input_path,
        index_col=0,
    )

    pred_df.index = (
        pd.to_datetime(
            pred_df.index,
            utc=True,
        )
        .tz_convert(
            "Europe/Copenhagen"
        )
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    target_col = (
        f"spread_{zone_a}_{zone_b}"
    )

    if target_col not in historical_df.columns:

        raise KeyError(
            f"{target_col} not found "
            f"in clean historical dataset."
        )

    historical_spread = (
        historical_df[target_col]
        .dropna()
    )

    # ---------------------------------------------------------------
    # NTC
    # ---------------------------------------------------------------

    code_a = ZONES[zone_a]
    code_b = ZONES[zone_b]

    ntc_fwd_col = (
        f"ntc_{code_a}_to_{code_b}"
    )

    ntc_rev_col = (
        f"ntc_{code_b}_to_{code_a}"
    )

    if (
        ntc_fwd_col in historical_df.columns
        and ntc_rev_col in historical_df.columns
        and historical_df[
            ntc_fwd_col
        ].notna().any()
    ):

        ntc_forward = (
            historical_df[
                ntc_fwd_col
            ]
            .sort_index()
        )

        ntc_reverse = (
            historical_df[
                ntc_rev_col
            ]
            .sort_index()
        )

        ntc_status = (
            "published NTC available"
        )

    else:

        ntc_forward = None
        ntc_reverse = None

        ntc_status = (
            "NO NTC series — "
            "stress uses explicit hard capacity ceiling"
        )

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"PHASE 5B — EXECUTION ROBUSTNESS: "
        f"{zone_a} <--> {zone_b}"
    )

    print(
        f"NTC status: {ntc_status}"
    )

    print(
        "=" * 90
    )

    # ---------------------------------------------------------------

    # ---------------------------------------------------------------

    for scenario_name, scenario in STRESS_SCENARIOS.items():

        results_df, pnl_df, theta_df = (
            run_stress_backtest(
                pred_df=pred_df,
                historical_spread=historical_spread,
                scenario=scenario,
                ntc_forward=ntc_forward,
                ntc_reverse=ntc_reverse,
            )
        )

        results_df["border"] = (
            f"{zone_a}_{zone_b}"
        )

        results_df["scenario"] = (
            scenario_name
        )

        all_stress_results.append(
            results_df
        )

        # -----------------------------------------------------------

        # -----------------------------------------------------------

        results_path = (
            DATA_DIR
            / f"phase5_execution_stress_"
              f"{scenario_name}_{suffix}.csv"
        )

        pnl_path = (
            DATA_DIR
            / f"phase5_execution_stress_pnl_"
              f"{scenario_name}_{suffix}.csv"
        )

        theta_path = (
            DATA_DIR
            / f"phase5_execution_stress_theta_"
              f"{scenario_name}_{suffix}.csv"
        )

        hac_path = (
            DATA_DIR
            / f"phase5_execution_stress_hac_"
              f"{scenario_name}_{suffix}.csv"
        )

        results_df.to_csv(
            results_path,
            index=False,
        )

        pnl_df.to_csv(
            pnl_path
        )

        theta_df.to_csv(
            theta_path,
            index=False,
        )

        hac_df = (
            run_stress_hac_sensitivity(
                pnl_df
            )
        )

        hac_df.to_csv(
            hac_path,
            index=False,
        )

        # -----------------------------------------------------------

        # -----------------------------------------------------------

        print(
            f"\nSCENARIO: {scenario_name}"
        )

        print(
            "  "
            f"fees="
            f"{scenario['transaction_cost_eur_per_mwh']:.2f} "
            "€/MWh total, "
            f"slippage="
            f"{scenario['slippage_base_eur_per_mwh']:.2f} "
            "€/MWh total at reference volatility, "
            f"BRP haircut="
            f"{scenario['brp_haircut']:.0%}, "
            f"NTC cap="
            f"{scenario['ntc_utilization_cap']:.1%}, "
            f"no-NTC stress cap="
            f"{scenario['no_ntc_hard_cap_mwh']:.0f} MWh"
        )

        style_table(
            results_df[["strategy", "cumulative_pnl_eur", "naive_daily_sharpe",
                       "hac_daily_sharpe_7d", "hac_daily_sharpe_30d", "max_drawdown_eur",
                       "win_rate", "active_share", "avg_abs_position_units",
                       "total_slippage_cost_eur", "brp_haircut", "ntc_utilization_cap",
                       "no_ntc_hard_cap_mwh"]],
            caption=f"{scenario_name} stress scenario — {zone_a}-{zone_b}",
            money_cols=["cumulative_pnl_eur", "max_drawdown_eur", "total_slippage_cost_eur"],
            pct_cols=["win_rate", "active_share", "brp_haircut", "ntc_utilization_cap"],
            highlight_max_cols=["cumulative_pnl_eur", "hac_daily_sharpe_7d"],
        )


# -----------------------------------------------------------------------
# COMBINE ALL OUTPUTS
# -----------------------------------------------------------------------

if all_stress_results:

    stress_results_all = pd.concat(
        all_stress_results,
        ignore_index=True,
    )

    stress_results_all.to_csv(
        DATA_DIR
        / "phase5_execution_stress_all_borders.csv",
        index=False,
    )

else:

    stress_results_all = (
        pd.DataFrame()
    )


print(
    "\nPHASE 5B COMPLETE"
)

print(
    "Baseline Phase 5 results are intentionally unchanged."
)

print(
    "Phase 5B adds conservative execution stress tests only."
)


PHASE 5B — EXECUTION ROBUSTNESS: DK1 <--> DE_LU
NTC status: published NTC available

SCENARIO: moderate_stress
  fees=0.16 €/MWh total, slippage=1.00 €/MWh total at reference volatility, BRP haircut=20%, NTC cap=7.5%, no-NTC stress cap=75 MWh


strategy,cumulative_pnl_eur,naive_daily_sharpe,hac_daily_sharpe_7d,hac_daily_sharpe_30d,max_drawdown_eur,win_rate,active_share,avg_abs_position_units,total_slippage_cost_eur,brp_haircut,ntc_utilization_cap,no_ntc_hard_cap_mwh
persistence_stress,"€3,449,651",8.211,6.270,6.304,"€-40,568",47.8%,35.2%,0.195,"€660,863",20.0%,7.5%,75.000
seasonal_naive_stress,"€2,181,324",6.565,4.765,4.312,"€-77,823",36.5%,35.7%,0.193,"€671,057",20.0%,7.5%,75.000
elastic_net_stress,"€5,954,901",9.964,6.877,6.694,"€-32,017",43.8%,64.3%,0.366,"€1,204,603",20.0%,7.5%,75.000
lightgbm_point_stress,"€6,137,159",10.432,7.158,6.965,"€-31,568",49.1%,57.4%,0.334,"€1,073,481",20.0%,7.5%,75.000
lstm_stress,"€5,429,405",9.537,6.487,6.232,"€-20,721",55.5%,39.4%,0.221,"€740,313",20.0%,7.5%,75.000
lightgbm_confidence_weighted_stress,"€4,342,481",8.721,5.530,4.557,"€-54,527",61.1%,30.0%,0.172,"€530,135",20.0%,7.5%,75.000



SCENARIO: conservative_stress
  fees=0.22 €/MWh total, slippage=1.50 €/MWh total at reference volatility, BRP haircut=30%, NTC cap=5.0%, no-NTC stress cap=50 MWh


strategy,cumulative_pnl_eur,naive_daily_sharpe,hac_daily_sharpe_7d,hac_daily_sharpe_30d,max_drawdown_eur,win_rate,active_share,avg_abs_position_units,total_slippage_cost_eur,brp_haircut,ntc_utilization_cap,no_ntc_hard_cap_mwh
persistence_stress,"€2,571,143",7.188,5.596,5.711,"€-36,261",44.1%,35.2%,0.194,"€988,147",30.0%,5.0%,50.000
seasonal_naive_stress,"€1,459,976",5.123,3.753,3.502,"€-94,740",33.2%,35.7%,0.192,"€1,003,447",30.0%,5.0%,50.000
elastic_net_stress,"€4,364,767",8.648,5.966,5.772,"€-56,550",43.9%,53.5%,0.302,"€1,494,275",30.0%,5.0%,50.000
lightgbm_point_stress,"€4,689,305",9.408,6.480,6.280,"€-52,899",48.6%,51.1%,0.296,"€1,428,974",30.0%,5.0%,50.000
lstm_stress,"€4,249,872",8.772,5.964,5.738,"€-24,973",51.9%,39.4%,0.220,"€1,106,162",30.0%,5.0%,50.000
lightgbm_confidence_weighted_stress,"€3,442,443",8.167,5.208,4.352,"€-71,385",56.9%,30.0%,0.171,"€790,884",30.0%,5.0%,50.000



SCENARIO: severe_stress
  fees=0.30 €/MWh total, slippage=2.00 €/MWh total at reference volatility, BRP haircut=30%, NTC cap=2.5%, no-NTC stress cap=25 MWh


...


SCENARIO: conservative_stress
  fees=0.22 €/MWh total, slippage=1.50 €/MWh total at reference volatility, BRP haircut=30%, NTC cap=5.0%, no-NTC stress cap=50 MWh


strategy,cumulative_pnl_eur,naive_daily_sharpe,hac_daily_sharpe_7d,hac_daily_sharpe_30d,max_drawdown_eur,win_rate,active_share,avg_abs_position_units,total_slippage_cost_eur,brp_haircut,ntc_utilization_cap,no_ntc_hard_cap_mwh
persistence_stress,"€8,799,926",18.413,9.453,6.509,"€-26,576",68.4%,78.9%,0.223,"€2,241,806",30.0%,5.0%,50.000
seasonal_naive_stress,"€7,464,192",15.438,7.827,4.837,"€-194,463",63.9%,77.7%,0.218,"€2,208,712",30.0%,5.0%,50.000
elastic_net_stress,"€9,651,566",18.181,9.680,6.356,"€-172,057",67.0%,87.6%,0.250,"€2,484,320",30.0%,5.0%,50.000
lightgbm_point_stress,"€10,320,738",20.054,11.190,7.945,"€-24,375",68.7%,91.3%,0.263,"€2,587,455",30.0%,5.0%,50.000
lstm_stress,"€9,660,472",19.259,10.049,6.716,"€-97,297",69.0%,85.9%,0.245,"€2,432,996",30.0%,5.0%,50.000
lightgbm_confidence_weighted_stress,"€10,163,917",20.569,10.869,7.449,"€-59,957",71.5%,83.8%,0.231,"€2,269,733",30.0%,5.0%,50.000



SCENARIO: severe_stress
  fees=0.30 €/MWh total, slippage=2.00 €/MWh total at reference volatility, BRP haircut=30%, NTC cap=2.5%, no-NTC stress cap=25 MWh


strategy,cumulative_pnl_eur,naive_daily_sharpe,hac_daily_sharpe_7d,hac_daily_sharpe_30d,max_drawdown_eur,win_rate,active_share,avg_abs_position_units,total_slippage_cost_eur,brp_haircut,ntc_utilization_cap,no_ntc_hard_cap_mwh
persistence_stress,"€7,254,450",16.974,8.426,5.492,"€-34,369",68.3%,75.6%,0.179,"€2,526,457",30.0%,2.5%,25.000
seasonal_naive_stress,"€6,175,296",14.371,7.108,4.285,"€-187,939",63.3%,74.8%,0.177,"€2,503,515",30.0%,2.5%,25.000
elastic_net_stress,"€7,915,961",16.985,8.588,5.375,"€-169,866",65.1%,87.6%,0.208,"€2,892,954",30.0%,2.5%,25.000
lightgbm_point_stress,"€8,487,530",18.832,9.823,6.377,"€-25,638",69.1%,86.5%,0.205,"€2,834,529",30.0%,2.5%,25.000
lstm_stress,"€7,936,354",17.920,8.930,5.663,"€-73,168",70.4%,78.3%,0.185,"€2,599,969",30.0%,2.5%,25.000
lightgbm_confidence_weighted_stress,"€8,352,065",18.960,9.551,6.108,"€-72,642",70.4%,81.4%,0.188,"€2,584,722",30.0%,2.5%,25.000



PHASE 5B COMPLETE
Baseline Phase 5 results are intentionally unchanged.
Phase 5B adds conservative execution stress tests only.


In [8]:
"""P&L and Serial Dependence Diagnostics Diagnoses whether a border's high Sharpe reflects genuine serial structure in that spread's P&L versus an artifact of treating autocorrelated hourly P&L as if it were closer to i.i.d."""

import numpy as np
import pandas as pd
from pathlib import Path

try:
    from IPython.display import display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False


def style_table(df: pd.DataFrame, caption: str = None, money_cols: list[str] = None,
                pct_cols: list[str] = None, highlight_max_cols: list[str] = None,
                highlight_min_cols: list[str] = None, decimals: int = 3):
    """Styling helper duplicated across phases (not imported) so each cell
    stays independently runnable."""
    money_cols = money_cols or []
    pct_cols = pct_cols or []
    highlight_max_cols = highlight_max_cols or []
    highlight_min_cols = highlight_min_cols or []

    fmt = {}
    for c in df.columns:
        if c in money_cols:
            fmt[c] = "€{:,.0f}"
        elif c in pct_cols:
            fmt[c] = "{:.1%}"
        elif pd.api.types.is_float_dtype(df[c]):
            fmt[c] = f"{{:.{decimals}f}}"

    styler = df.style.format(fmt, na_rep="-").hide(axis="index")
    if caption:
        styler = styler.set_caption(caption)
    for c in highlight_max_cols:
        if c in df.columns:
            styler = styler.highlight_max(subset=[c], color="#c6efce")
    for c in highlight_min_cols:
        if c in df.columns:
            styler = styler.highlight_min(subset=[c], color="#ffc7ce")

    if HAS_IPYTHON:
        display(styler)
    else:
        print(f"\n{caption or ''}")
        print(df.to_string(index=False))
    return styler

# -----------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------

AUTOCORR_LAGS_HOURS = [1, 2, 6, 12, 24, 48, 72, 168]
HAC_SHARPE_LAG_DAYS = [1, 7, 14, 30, 60, 90]


def autocorr_at_lag(hourly_pnl: pd.Series, lag: int) -> float:
    """Pearson autocorrelation of hourly P&L at a given hourly lag."""
    if len(hourly_pnl) <= lag + 1:
        return float("nan")
    return float(hourly_pnl.autocorr(lag=lag))


def hac_long_run_variance(x: np.ndarray, q: int) -> float:
    """Bartlett-kernel Newey-West long-run variance -- same construction as inside sharpe_ratio() in baseline Phase 5, factored out here so it can..."""
    mu = x.mean()
    centered = x - mu
    gamma0 = np.mean(centered ** 2)
    if gamma0 <= 0:
        return 0.0
    q = min(q, len(x) - 1)
    long_run = gamma0
    for k in range(1, q + 1):
        weight = 1.0 - k / (q + 1.0)
        cov = np.mean(centered[k:] * centered[:-k])
        long_run += 2.0 * weight * cov
    return float(long_run)


def effective_independent_days(daily_pnl: pd.Series, q: int) -> float:
    """Newey-West-implied effective sample size: N * (short-run variance / HAC long-run variance) at lag q."""
    x = daily_pnl.to_numpy(dtype=float)
    n = len(x)
    if n < max(20, q + 2):
        return float("nan")
    mu = x.mean()
    gamma0 = np.mean((x - mu) ** 2)
    long_run = hac_long_run_variance(x, q)
    if long_run <= 0:
        return float("nan")
    return float(n * gamma0 / long_run)


def diagnose_strategy(strategy_name: str, cum_pnl: pd.Series) -> dict:
    cum_pnl = cum_pnl.sort_index().dropna()
    if len(cum_pnl) < 2:
        return {"strategy": strategy_name}

    hourly_pnl = cum_pnl.diff()
    hourly_pnl.iloc[0] = cum_pnl.iloc[0]

    daily_pnl = hourly_pnl.resample("D").sum().dropna()
    weekly_pnl = hourly_pnl.resample("W").sum().dropna()
    monthly_pnl = hourly_pnl.resample("ME").sum().dropna()

    row = {
        "strategy": strategy_name,

        "hourly_mean_pnl_eur": float(hourly_pnl.mean()),
        "hourly_std_pnl_eur": float(hourly_pnl.std()),

        "daily_mean_pnl_eur": float(daily_pnl.mean()) if len(daily_pnl) else float("nan"),
        "daily_std_pnl_eur": float(daily_pnl.std()) if len(daily_pnl) else float("nan"),

        "weekly_mean_pnl_eur": float(weekly_pnl.mean()) if len(weekly_pnl) else float("nan"),
        "weekly_std_pnl_eur": float(weekly_pnl.std()) if len(weekly_pnl) else float("nan"),

        "monthly_mean_pnl_eur": float(monthly_pnl.mean()) if len(monthly_pnl) else float("nan"),
        "monthly_std_pnl_eur": float(monthly_pnl.std()) if len(monthly_pnl) else float("nan"),

        "cumulative_pnl_eur": float(cum_pnl.iloc[-1]),
        "max_drawdown_eur": max_drawdown(cum_pnl),
        "avg_daily_pnl_eur": float(daily_pnl.mean()) if len(daily_pnl) else float("nan"),
        "pct_profitable_days": float((daily_pnl > 0).mean()) if len(daily_pnl) else float("nan"),
        "n_days": int(len(daily_pnl)),
    }


    for lag in AUTOCORR_LAGS_HOURS:
        row[f"autocorr_hourly_lag_{lag}h"] = autocorr_at_lag(hourly_pnl, lag)


    for lag in HAC_SHARPE_LAG_DAYS:
        row[f"hac_sharpe_{lag}d"] = sharpe_ratio(hourly_pnl, max_lag_days=lag)
        row[f"effective_independent_days_{lag}d"] = effective_independent_days(daily_pnl, lag)

    return row


# -----------------------------------------------------------------------
# RUN FOR EVERY BORDER / STRATEGY
# -----------------------------------------------------------------------

all_diagnostics = []

for zone_a, zone_b in BORDERS:
    suffix = f"{zone_a}_{zone_b}"
    pnl_path = DATA_DIR / f"phase5_pnl_curves_{suffix}.csv"

    if not pnl_path.exists():
        print(f"WARNING: {pnl_path} not found — skipping {zone_a}-{zone_b} "
              "(run baseline Phase 5 first)")
        continue

    pnl_curves = pd.read_csv(pnl_path, index_col=0)
    pnl_curves.index = pd.to_datetime(pnl_curves.index, utc=True).tz_convert("Europe/Copenhagen")

    print("\n" + "=" * 90)
    print(f"PHASE 5C — P&L / SERIAL DEPENDENCE DIAGNOSTICS: {zone_a} <--> {zone_b}")
    print("=" * 90)

    border_rows = []
    for strategy_name in pnl_curves.columns:
        row = diagnose_strategy(strategy_name, pnl_curves[strategy_name])
        row["border"] = suffix
        border_rows.append(row)
        all_diagnostics.append(row)

    border_df = pd.DataFrame(border_rows)

    summary_cols = ["strategy", "cumulative_pnl_eur", "max_drawdown_eur",
                     "avg_daily_pnl_eur", "pct_profitable_days", "n_days"] +\
                    [f"hac_sharpe_{l}d" for l in HAC_SHARPE_LAG_DAYS]
    style_table(
        border_df[summary_cols],
        caption=f"HAC Sharpe across lags — {zone_a}-{zone_b} (watch for Sharpe falling as lag grows)",
        money_cols=["cumulative_pnl_eur", "max_drawdown_eur", "avg_daily_pnl_eur"],
        pct_cols=["pct_profitable_days"],
    )

    autocorr_cols = ["strategy"] + [f"autocorr_hourly_lag_{l}h" for l in AUTOCORR_LAGS_HOURS]
    style_table(
        border_df[autocorr_cols],
        caption=f"Hourly P&L autocorrelation — {zone_a}-{zone_b} (watch for persistently large values out to 24h/48h/168h)",
    )

    eff_n_cols = ["strategy"] + [f"effective_independent_days_{l}d" for l in HAC_SHARPE_LAG_DAYS]
    style_table(
        border_df[eff_n_cols],
        caption=f"Effective independent trading days by HAC lag — {zone_a}-{zone_b} (vs. raw n_days above)",
        decimals=1,
    )

    border_df.to_csv(DATA_DIR / f"phase5c_diagnostics_{suffix}.csv", index=False)

diagnostics_all = pd.DataFrame(all_diagnostics)
if len(diagnostics_all):
    diagnostics_all.to_csv(DATA_DIR / "phase5c_diagnostics_all_borders.csv", index=False)

print("\nPHASE 5C COMPLETE")


PHASE 5C — P&L / SERIAL DEPENDENCE DIAGNOSTICS: DK1 <--> DE_LU


strategy,cumulative_pnl_eur,max_drawdown_eur,avg_daily_pnl_eur,pct_profitable_days,n_days,hac_sharpe_1d,hac_sharpe_7d,hac_sharpe_14d,hac_sharpe_30d,hac_sharpe_60d,hac_sharpe_90d
persistence_fold_tuned,"€4,323,182","€-44,887","€5,305",49.3%,815,7.545,6.743,6.589,6.701,7.086,7.465
seasonal_naive_fold_tuned,"€2,900,959","€-71,938","€3,559",43.1%,815,6.427,5.488,5.126,4.857,4.825,5.099
elastic_net_fold_tuned,"€7,652,063","€-21,628","€9,389",61.8%,815,9.196,7.714,7.411,7.470,7.957,8.064
lightgbm_point_fold_tuned,"€7,621,366","€-17,807","€9,351",60.6%,815,9.294,7.744,7.503,7.572,8.060,8.226
lstm_fold_tuned,"€6,604,499","€-16,229","€8,104",52.0%,815,8.311,6.860,6.609,6.569,6.843,6.845
lightgbm_confidence_weighted,"€5,241,830","€-45,638","€6,432",43.7%,815,7.437,5.753,5.175,4.688,4.135,3.869


strategy,autocorr_hourly_lag_1h,autocorr_hourly_lag_2h,autocorr_hourly_lag_6h,autocorr_hourly_lag_12h,autocorr_hourly_lag_24h,autocorr_hourly_lag_48h,autocorr_hourly_lag_72h,autocorr_hourly_lag_168h
persistence_fold_tuned,0.784,0.586,0.255,0.218,0.237,0.047,0.018,0.013
seasonal_naive_fold_tuned,0.707,0.502,0.239,0.182,0.137,0.054,0.017,0.108
elastic_net_fold_tuned,0.868,0.706,0.371,0.320,0.240,0.077,0.037,0.016
lightgbm_point_fold_tuned,0.863,0.700,0.362,0.322,0.238,0.080,0.041,0.007
lstm_fold_tuned,0.863,0.701,0.366,0.337,0.250,0.078,0.041,0.017
lightgbm_confidence_weighted,0.845,0.682,0.363,0.343,0.266,0.115,0.060,0.042


strategy,effective_independent_days_1d,effective_independent_days_7d,effective_independent_days_14d,effective_independent_days_30d,effective_independent_days_60d,effective_independent_days_90d
persistence_fold_tuned,587.2,468.9,447.8,463.2,517.9,574.8
seasonal_naive_fold_tuned,606.8,442.5,386.1,346.6,342.0,382.0
elastic_net_fold_tuned,567.7,399.4,368.7,374.6,425.0,436.5
lightgbm_point_fold_tuned,563.6,391.4,367.3,374.2,423.9,441.6
lstm_fold_tuned,566.1,385.6,357.9,353.6,383.8,384.0
lightgbm_confidence_weighted,551.4,329.9,267.0,219.1,170.5,149.3



PHASE 5C — P&L / SERIAL DEPENDENCE DIAGNOSTICS: DK1 <--> DK2


...


PHASE 5C — P&L / SERIAL DEPENDENCE DIAGNOSTICS: DK1 <--> SE3


strategy,cumulative_pnl_eur,max_drawdown_eur,avg_daily_pnl_eur,pct_profitable_days,n_days,hac_sharpe_1d,hac_sharpe_7d,hac_sharpe_14d,hac_sharpe_30d,hac_sharpe_60d,hac_sharpe_90d
persistence_fold_tuned,"€13,589,913","€-32,332","€16,675",87.1%,815,16.572,10.951,9.358,7.546,6.324,5.750
seasonal_naive_fold_tuned,"€11,887,088","€-110,669","€14,585",84.2%,815,14.884,9.395,7.447,5.796,4.719,4.301
elastic_net_fold_tuned,"€14,959,380","€-98,578","€18,355",87.5%,815,17.111,11.453,9.631,7.598,6.030,5.372
lightgbm_point_fold_tuned,"€15,913,515","€-24,491","€19,526",90.7%,815,18.967,13.161,11.433,9.398,7.789,7.097
lstm_fold_tuned,"€14,970,669","€-120,781","€18,369",90.3%,815,17.967,11.806,9.902,7.877,6.471,5.915
lightgbm_confidence_weighted,"€15,621,230","€-38,292","€19,167",91.5%,815,18.645,12.574,10.736,8.672,7.062,6.357


strategy,autocorr_hourly_lag_1h,autocorr_hourly_lag_2h,autocorr_hourly_lag_6h,autocorr_hourly_lag_12h,autocorr_hourly_lag_24h,autocorr_hourly_lag_48h,autocorr_hourly_lag_72h,autocorr_hourly_lag_168h
persistence_fold_tuned,0.827,0.673,0.334,0.202,0.530,0.373,0.324,0.246
seasonal_naive_fold_tuned,0.818,0.681,0.313,0.173,0.402,0.335,0.317,0.371
elastic_net_fold_tuned,0.901,0.767,0.391,0.259,0.463,0.361,0.310,0.264
lightgbm_point_fold_tuned,0.900,0.762,0.390,0.253,0.442,0.333,0.286,0.228
lstm_fold_tuned,0.898,0.761,0.386,0.254,0.459,0.357,0.336,0.277
lightgbm_confidence_weighted,0.905,0.776,0.404,0.286,0.489,0.376,0.329,0.264


strategy,effective_independent_days_1d,effective_independent_days_7d,effective_independent_days_14d,effective_independent_days_30d,effective_independent_days_60d,effective_independent_days_90d
persistence_fold_tuned,491.2,214.5,156.7,101.8,71.5,59.1
seasonal_naive_fold_tuned,528.4,210.5,132.3,80.1,53.1,44.1
elastic_net_fold_tuned,520.1,233.0,164.8,102.6,64.6,51.3
lightgbm_point_fold_tuned,532.8,256.5,193.6,130.8,89.8,74.6
lstm_fold_tuned,520.7,224.8,158.2,100.1,67.6,56.4
lightgbm_confidence_weighted,513.4,233.5,170.2,111.1,73.7,59.7



PHASE 5C COMPLETE


# Model Explainability & Economic Interpretation

SHAP is used descriptively to show which features drive the final LightGBM predictions. It is not treated as causal evidence or as formal validation of an economic mechanism.


In [9]:
"""Phase 6: SHAP Analysis (descriptive economic interpretation only)"""
import shap
import matplotlib.pyplot as plt
import lightgbm as lgb
import pandas as pd
from pathlib import Path

ZONES = {"DK1": "DK_1", "DE_LU": "DE_LU", "DK2": "DK_2", "SE3": "SE_3"}
BORDERS = [("DK1", "DE_LU"), ("DK1", "DK2"), ("DK1", "SE3")]


def get_feature_columns(df: pd.DataFrame) -> list[str]:
    exclude = {f"spread_{a}_{b}" for a, b in BORDERS} | {f"price_{code}" for code in ZONES.values()}
    return [c for c in df.columns if c not in exclude]


def add_interaction_terms(df, base_features, zone_a, zone_b):
    df = df.copy()
    new_cols = []
    code_a, code_b = ZONES[zone_a], ZONES[zone_b]
    wind_diff_col = f"delta_wind_{zone_a}_{zone_b}"
    ntc_col = f"ntc_{code_a}_to_{code_b}"
    resdem_a_col = f"residual_demand_{zone_a}"
    resdem_b_col = f"residual_demand_{zone_b}"
    if ntc_col in df.columns:
        if wind_diff_col in df.columns:
            df["interact_wind_ntc"] = df[wind_diff_col] * df[ntc_col]
            new_cols.append("interact_wind_ntc")
        if resdem_a_col in df.columns and resdem_b_col in df.columns:
            df["residual_demand_diff"] = df[resdem_a_col] - df[resdem_b_col]
            df["interact_resdemand_ntc"] = df["residual_demand_diff"] * df[ntc_col]
            new_cols.extend(["residual_demand_diff", "interact_resdemand_ntc"])
    return df, base_features + new_cols


def run_shap_analysis():
    data_path = Path("./data/model_ready_dataset.parquet")
    df_full = pd.read_parquet(data_path)
    base_features = get_feature_columns(df_full)
    for zone_a, zone_b in BORDERS:
        print(f"\nAnalyzing {zone_a}-{zone_b}...")
        df_border, feature_cols = add_interaction_terms(df_full, base_features, zone_a, zone_b)
        X = df_border[feature_cols].tail(max(1, int(len(df_border) * 0.20)))
        model_path = Path(f"./models/lightgbm_point_final_{zone_a}_{zone_b}.txt")
        if not model_path.exists():
            print(f"Model {model_path} missing. Skipping.")
            continue
        model = lgb.Booster(model_file=str(model_path))
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        shap.summary_plot(shap_values, X, show=False)
        plt.savefig(f"shap_summary_{zone_a}_{zone_b}.png", bbox_inches="tight")
        plt.close()
        print(f"SHAP successfully computed for {len(feature_cols)} features on the latest {len(X)} observations; descriptive only.")

if __name__ == "__main__":
    run_shap_analysis()



Analyzing DK1-DE_LU...
SHAP successfully computed for 57 features on the latest 6394 observations; descriptive only.

Analyzing DK1-DK2...
SHAP successfully computed for 54 features on the latest 6394 observations; descriptive only.

Analyzing DK1-SE3...
SHAP successfully computed for 54 features on the latest 6394 observations; descriptive only.


# Lag-Robustness Check: Day-Ahead Information-Set Sensitivity Test

In [ ]:
"""
Diagnostic script -- run this with your real ENTSOE_API_KEY.
Does NOT touch your main pipeline. Fetches ONE small raw response and
inspects exactly what metadata is actually present, and whether it's
useful (per-day granular) or not (one stale value for the whole query).
"""
import os
from entsoe import EntsoeRawClient
import pandas as pd
import bs4

client = EntsoeRawClient(api_key=os.environ["ENTSOE_API_KEY"])

# Small, cheap query: 5 days of DK1 load forecast.
start = pd.Timestamp("2023-01-01", tz="Europe/Copenhagen")
end = pd.Timestamp("2023-01-06", tz="Europe/Copenhagen")

raw_xml = client.query_load_forecast("DK_1", start=start, end=end)

soup = bs4.BeautifulSoup(raw_xml, "html.parser")

# 1. Is there a document-level createdDateTime, and how many?
created = soup.find_all("createddatetime")
print(f"Number of createdDateTime tags found: {len(created)}")
for c in created:
    print("  ", c.text)

# 2. Is there a document-level revisionNumber, and how many?
revisions = soup.find_all("revisionnumber")
print(f"\nNumber of revisionNumber tags found: {len(revisions)}")
for r in revisions:
    print("  ", r.text)

# 3. How many separate TimeSeries blocks came back for these 5 days?
series = soup.find_all("timeseries")
print(f"\nNumber of <TimeSeries> blocks for a 5-day query: {len(series)}")

# 4. THE KEY CHECK: is createdDateTime INSIDE each TimeSeries block
#    (per-day, potentially useful), or only at the document level
#    (one value for the whole query, likely NOT useful)?
for i, ts in enumerate(series):
    ts_created = ts.find("createddatetime")
    ts_revision = ts.find("revisionnumber")
    period_start = ts.find("start")
    print(f"  TimeSeries {i}: period_start={period_start.text if period_start else None}, "
          f"own createdDateTime={ts_created.text if ts_created else 'NONE (inherits document-level only)'}, "
          f"own revisionNumber={ts_revision.text if ts_revision else 'NONE'}")

A diagnostic query against the live ENTSO-E API (see the cell above) confirmed
that the platform's `createdDateTime` field reflects when an API response was
generated, not when the underlying forecast was originally published -- so
there is no accessible metadata that verifies whether the load, wind/solar,
and NTC forecast features used throughout this pipeline were genuinely
available at the day-ahead decision time they are assumed to represent.

Rather than leave that assumption unexamined, this section tests its
consequences directly: Elastic Net and LightGBM Point (the two models
carrying this thesis's central forecasting claims) are re-estimated at all
three borders with every forecast-derived feature -- load, wind, solar, NTC,
and their interaction terms -- shifted an *additional* 24 hours further back
in time, on top of whatever causal alignment the main pipeline already
applies. Persistence and Seasonal Naive are excluded, since they use only
the target's own lagged values and are mechanically unaffected by this
shift; the LSTM is excluded for runtime, since it is not central to the
thesis's headline RMSE/DM-test claims.

If the main pipeline's results depended on forecast information that was not
genuinely available day-ahead, this deliberately conservative extra lag
should degrade performance. If results are unaffected, that is direct
evidence against look-ahead bias as an explanation for the reported
findings -- not proof the assumption holds for every observation, but a
concrete test of whether it matters.

In [11]:
"""
Lag-Robustness Check: Day-Ahead Information-Set Sensitivity Test
==================================================================
Tests whether the thesis's core forecasting results are sensitive to the
unverifiable assumption that ENTSO-E's day-ahead forecast products (load
forecast, wind/solar forecast, NTC) were genuinely available at the
day-ahead decision time.

A diagnostic query already confirmed ENTSO-E's API exposes no per-
observation publication-vintage metadata (createdDateTime reflects the API
response time, not the original forecast publication time) -- so instead
of trying to verify the assumption, this asks: if it were wrong by an
additional 24 hours, how much would headline results actually change?

SCOPE (deliberately minimal, for runtime): only Elastic Net and LightGBM
Point are re-run -- these carry the thesis's central forecasting claims.
Persistence and Seasonal Naive are excluded: they use only the target's
own lagged values, never the forecast features, so they are mechanically
unaffected and re-running them is uninformative. LSTM is excluded purely
for runtime (30+ min/border in the full pipeline); it is not central to
the thesis's headline RMSE/DM claims, and if the two fast models turn out
sensitive, LSTM can be added as a targeted follow-up rather than paid for
upfront on every run.
"""

import logging
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

try:
    from IPython.display import display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

DATA_DIR = Path("./data")
INPUT_PATH = DATA_DIR / "model_ready_dataset.parquet"

ZONES = {"DK1": "DK_1", "DE_LU": "DE_LU", "DK2": "DK_2", "SE3": "SE_3"}
BORDERS = [("DK1", "DE_LU"), ("DK1", "DK2"), ("DK1", "SE3")]
TARGET_COLS = [f"spread_{a}_{b}" for a, b in BORDERS]

N_FOLDS = 5
MIN_TRAIN_FRAC = 0.4
EMBARGO_HOURS = 168
EN_CV_SPLITS = 5
SEED = 42

EXTRA_LAG_HOURS = 24  # the additional, deliberately conservative buffer


def style_table(df, caption=None, decimals=3):
    fmt = {c: f"{{:.{decimals}f}}" for c in df.columns if pd.api.types.is_float_dtype(df[c])}
    styler = df.style.format(fmt, na_rep="-").hide(axis="index")
    if caption:
        styler = styler.set_caption(caption)
    if HAS_IPYTHON:
        display(styler)
    else:
        print(f"\n{caption or ''}")
        print(df.to_string(index=False))
    return styler


def get_feature_columns(df):
    exclude = set(TARGET_COLS) | {f"price_{code}" for code in ZONES.values()}
    return [c for c in df.columns if c not in exclude]


def identify_forecast_dependent_columns(feature_cols):
    """Columns derived from ENTSO-E day-ahead FORECAST products (load,
    wind, solar, NTC, and their interaction terms) -- these carry the
    unverifiable publication-vintage assumption. Calendar features,
    autoregressive spread lags, and commodity proxies (already
    conservatively lagged in Phase 2) are deliberately excluded -- they
    either have no publication-timing ambiguity (calendar) or are handled
    elsewhere."""
    prefixes = ("load_forecast_", "wind_total_", "delta_wind_",
               "residual_demand_", "ntc_", "interact_")
    return [c for c in feature_cols if c.startswith(prefixes)]


def apply_extra_lag(df, cols, extra_hours=EXTRA_LAG_HOURS):
    """Shifts the given columns an ADDITIONAL `extra_hours` further back,
    on top of whatever causal alignment Phase 3 already applied."""
    df = df.copy()
    for c in cols:
        df[c] = df[c].shift(extra_hours)
    return df


def add_interaction_terms(df, base_features, zone_a, zone_b):
    df = df.copy()
    new_cols = []
    code_a, code_b = ZONES[zone_a], ZONES[zone_b]
    wind_diff_col = f"delta_wind_{zone_a}_{zone_b}"
    ntc_col = f"ntc_{code_a}_to_{code_b}"
    resdem_a_col = f"residual_demand_{zone_a}"
    resdem_b_col = f"residual_demand_{zone_b}"
    if ntc_col in df.columns:
        if wind_diff_col in df.columns:
            df["interact_wind_ntc"] = df[wind_diff_col] * df[ntc_col]
            new_cols.append("interact_wind_ntc")
        if resdem_a_col in df.columns and resdem_b_col in df.columns:
            df["residual_demand_diff"] = df[resdem_a_col] - df[resdem_b_col]
            df["interact_resdemand_ntc"] = df["residual_demand_diff"] * df[ntc_col]
            new_cols.extend(["residual_demand_diff", "interact_resdemand_ntc"])
    return df, base_features + new_cols


def make_walk_forward_folds(n_rows, n_folds=N_FOLDS, min_train_frac=MIN_TRAIN_FRAC, embargo_hours=EMBARGO_HOURS):
    min_train_size = int(n_rows * min_train_frac)
    remaining = n_rows - min_train_size
    test_size = remaining // n_folds
    folds = []
    train_end = min_train_size
    for _ in range(n_folds):
        test_start = train_end
        test_end = min(test_start + test_size, n_rows)
        if test_start >= n_rows:
            break
        embargoed_train_end = max(0, train_end - embargo_hours)
        folds.append((slice(0, embargoed_train_end), slice(test_start, test_end)))
        train_end = test_end
    return folds


def directional_accuracy(y_true, y_pred):
    return float(np.mean(np.sign(y_true) == np.sign(y_pred)))


def directional_accuracy_threshold(y_true, y_pred, threshold=1.0):
    mask = np.abs(y_true) > threshold
    if not mask.any():
        return float("nan")
    return float(np.mean(np.sign(y_true[mask]) == np.sign(y_pred[mask])))


def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "directional_accuracy": directional_accuracy(y_true, y_pred),
        "directional_accuracy_gt_1": directional_accuracy_threshold(y_true, y_pred),
    }


def _split_for_validation(X_train, y_train, y_val_split=0.15):
    n_val = max(int(len(X_train) * y_val_split), EMBARGO_HOURS + 1)
    if len(X_train) <= n_val + EMBARGO_HOURS:
        raise ValueError("Training window too short for validation plus embargo.")
    val_start = len(X_train) - n_val
    fit_end = val_start - EMBARGO_HOURS
    return X_train.iloc[:fit_end], X_train.iloc[val_start:], y_train.iloc[:fit_end], y_train.iloc[val_start:]


def fit_elastic_net(X_train, y_train, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    model = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0], alphas=100,
        cv=TimeSeriesSplit(n_splits=EN_CV_SPLITS, gap=EMBARGO_HOURS),
        max_iter=20000, tol=1e-5, n_jobs=-1,
    )
    model.fit(X_train_s, y_train)
    return model.predict(X_test_s)


def fit_lightgbm_point(X_train, y_train, X_test, y_val_split=0.15):
    X_fit, X_val, y_fit, y_val = _split_for_validation(X_train, y_train, y_val_split)
    train_set = lgb.Dataset(X_fit, label=y_fit)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)
    params = {
        "objective": "regression", "metric": "rmse", "learning_rate": 0.05, "num_leaves": 31,
        "min_data_in_leaf": 20, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 5,
        "seed": SEED, "feature_fraction_seed": SEED, "bagging_seed": SEED, "data_random_seed": SEED,
        "deterministic": True, "force_col_wise": True, "verbose": -1,
    }
    model = lgb.train(params, train_set, valid_sets=[val_set], num_boost_round=1000,
                      callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    return model.predict(X_test, num_iteration=model.best_iteration)


def run_walk_forward_minimal(df, feature_cols, target_col, label):
    """Elastic Net + LightGBM Point only -- the two models carrying this
    thesis's central forecasting claims."""
    folds = make_walk_forward_folds(len(df))
    rows = []
    for fold_i, (train_idx, test_idx) in enumerate(folds):
        train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
        X_train, y_train = train_df[feature_cols], train_df[target_col]
        X_test, y_test = test_df[feature_cols], test_df[target_col]

        en_preds = fit_elastic_net(X_train, y_train, X_test)
        rows.append({"fold": fold_i, "model": "elastic_net", "scenario": label,
                     **evaluate(y_test.values, en_preds)})

        if HAS_LIGHTGBM:
            lgb_preds = fit_lightgbm_point(X_train, y_train, X_test)
            rows.append({"fold": fold_i, "model": "lightgbm_point", "scenario": label,
                         **evaluate(y_test.values, lgb_preds)})
    return pd.DataFrame(rows)


if __name__ == "__main__":
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"{INPUT_PATH} not found — run Phase 3 first.")
    full_df = pd.read_parquet(INPUT_PATH)

    all_results = []
    for zone_a, zone_b in BORDERS:
        target_col = f"spread_{zone_a}_{zone_b}"
        log.info(f"\n{'='*70}\nBorder {zone_a}-{zone_b}: lag-robustness check\n{'='*70}")

        base_features = get_feature_columns(full_df)
        df_baseline, feature_cols = add_interaction_terms(full_df, base_features, zone_a, zone_b)
        df_baseline = df_baseline.dropna(subset=feature_cols + [target_col])

        forecast_cols = identify_forecast_dependent_columns(feature_cols)
        log.info(f"[{target_col}] {len(forecast_cols)} forecast-dependent columns will get "
                 f"an extra {EXTRA_LAG_HOURS}h lag: {forecast_cols}")

        df_stressed = apply_extra_lag(df_baseline, forecast_cols, EXTRA_LAG_HOURS)
        df_stressed = df_stressed.dropna(subset=feature_cols + [target_col])

        log.info(f"[{target_col}] baseline rows={len(df_baseline)}, "
                 f"stressed rows={len(df_stressed)} (fewer due to extra lag)")

        res_baseline = run_walk_forward_minimal(df_baseline, feature_cols, target_col, "baseline")
        res_stressed = run_walk_forward_minimal(df_stressed, feature_cols, target_col,
                                                f"extra_{EXTRA_LAG_HOURS}h_lag")

        combined = pd.concat([res_baseline, res_stressed], ignore_index=True)
        combined["border"] = f"{zone_a}_{zone_b}"
        all_results.append(combined)

        summary = combined.groupby(["scenario", "model"])[["rmse", "mae", "directional_accuracy",
                                                            "directional_accuracy_gt_1"]].mean().reset_index()
        style_table(summary, caption=f"Lag-robustness check — {zone_a}-{zone_b} "
                                     f"(baseline vs. extra {EXTRA_LAG_HOURS}h lag)")

    all_results_df = pd.concat(all_results, ignore_index=True)
    all_results_df.to_csv(DATA_DIR / "lag_robustness_check_all_borders.csv", index=False)
    log.info(f"Saved -> {DATA_DIR / 'lag_robustness_check_all_borders.csv'}")

2026-08-21 16:05:46,471 | INFO | 
Border DK1-DE_LU: lag-robustness check
2026-08-21 16:05:46,520 | INFO | [spread_DK1_DE_LU] 20 forecast-dependent columns will get an extra 24h lag: ['load_forecast_DE_LU', 'load_forecast_DK_1', 'load_forecast_DK_2', 'load_forecast_SE_3', 'ntc_DK_1_to_DE_LU', 'ntc_DE_LU_to_DK_1', 'wind_total_DE_LU', 'wind_total_DK1', 'wind_total_DK2', 'wind_total_SE3', 'delta_wind_DK1_DE_LU', 'delta_wind_DK1_DK2', 'delta_wind_DK1_SE3', 'residual_demand_DE_LU', 'residual_demand_DK1', 'residual_demand_DK2', 'residual_demand_SE3', 'interact_wind_ntc', 'residual_demand_diff', 'interact_resdemand_ntc']
2026-08-21 16:05:46,592 | INFO | [spread_DK1_DE_LU] baseline rows=31972, stressed rows=31948 (fewer due to extra lag)
/opt/anaconda3/envs/my_env_311/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increas

scenario,model,rmse,mae,directional_accuracy,directional_accuracy_gt_1
baseline,elastic_net,18.545,11.705,0.391,0.924
baseline,lightgbm_point,17.657,9.417,0.393,0.926
extra_24h_lag,elastic_net,18.538,11.919,0.388,0.913
extra_24h_lag,lightgbm_point,17.241,9.860,0.397,0.935


2026-08-21 16:06:11,088 | INFO | 
Border DK1-DK2: lag-robustness check
2026-08-21 16:06:11,102 | INFO | [spread_DK1_DK2] 17 forecast-dependent columns will get an extra 24h lag: ['load_forecast_DE_LU', 'load_forecast_DK_1', 'load_forecast_DK_2', 'load_forecast_SE_3', 'ntc_DK_1_to_DE_LU', 'ntc_DE_LU_to_DK_1', 'wind_total_DE_LU', 'wind_total_DK1', 'wind_total_DK2', 'wind_total_SE3', 'delta_wind_DK1_DE_LU', 'delta_wind_DK1_DK2', 'delta_wind_DK1_SE3', 'residual_demand_DE_LU', 'residual_demand_DK1', 'residual_demand_DK2', 'residual_demand_SE3']
2026-08-21 16:06:11,124 | INFO | [spread_DK1_DK2] baseline rows=31972, stressed rows=31948 (fewer due to extra lag)
/opt/anaconda3/envs/my_env_311/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.036e+02, tolerance: 1.609e+02
  model = cd

scenario,model,rmse,mae,directional_accuracy,directional_accuracy_gt_1
baseline,elastic_net,15.022,8.615,0.248,0.570
baseline,lightgbm_point,15.480,8.733,0.230,0.502
extra_24h_lag,elastic_net,15.003,8.432,0.252,0.578
extra_24h_lag,lightgbm_point,15.594,8.609,0.248,0.545


2026-08-21 16:06:22,869 | INFO | 
Border DK1-SE3: lag-robustness check
2026-08-21 16:06:22,876 | INFO | [spread_DK1_SE3] 17 forecast-dependent columns will get an extra 24h lag: ['load_forecast_DE_LU', 'load_forecast_DK_1', 'load_forecast_DK_2', 'load_forecast_SE_3', 'ntc_DK_1_to_DE_LU', 'ntc_DE_LU_to_DK_1', 'wind_total_DE_LU', 'wind_total_DK1', 'wind_total_DK2', 'wind_total_SE3', 'delta_wind_DK1_DE_LU', 'delta_wind_DK1_DK2', 'delta_wind_DK1_SE3', 'residual_demand_DE_LU', 'residual_demand_DK1', 'residual_demand_DK2', 'residual_demand_SE3']
2026-08-21 16:06:22,891 | INFO | [spread_DK1_SE3] baseline rows=31972, stressed rows=31948 (fewer due to extra lag)
/opt/anaconda3/envs/my_env_311/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.204e+03, tolerance: 8.893e+02
  model = cd

scenario,model,rmse,mae,directional_accuracy,directional_accuracy_gt_1
baseline,elastic_net,31.715,23.674,0.743,0.850
baseline,lightgbm_point,29.872,21.034,0.793,0.909
extra_24h_lag,elastic_net,30.962,22.787,0.748,0.855
extra_24h_lag,lightgbm_point,28.238,20.242,0.790,0.905


2026-08-21 16:06:41,682 | INFO | Saved -> data/lag_robustness_check_all_borders.csv
